# GameTheory 3e : Meta-Actions Tarifees et Parcours Complet -- l'agent qui joue, change les regles, puis paie le trajet

[← GameTheory-03d-Deux-Especes-de-Fleches-Python](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) | [↑ README GameTheory](README.md) | [jumeau Python](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb)

Les notebooks [GT-3](GameTheory-03-Topology2x2-Python.ipynb), [GT-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) et [GT-3d](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) ont fait de l'espace des jeux 2x2 ordinaux un **univers fini manipulable** : 576 chambres strictes, six swaps generateurs, des murs ou vivent les egalites. Ce notebook monte l'echelle d'un cran : les swaps y deviennent des **actions que l'agent paie**.

## Prérequis
- [03-Topology2x2](GameTheory-03-Topology2x2-Python.ipynb) — l'univers fini des 576 jeux 2x2 ordinaux
## Plan

1. **Marche 1 -- l'agent joue** : equilibres de Nash purs et dynamique de meilleure reponse sur les 576 jeux
2. **Marche 2 -- l'agent deplace le jeu** : meta-actions tarifees, un cout mesure en echelons de rang, seuil de migration
3. **Marche 3 -- deux agents en desaccord** : le meta-jeu 4x4, ses equilibres, et la question de l'accord

## La question

La strate 6 de la theorie des jeux dit ce qu'un agent fait *dans* des regles donnees. La strate 7 demande ce qui se passe quand **modifier les regles est lui-meme une action** -- avec un cout, donc un arbitrage, donc un equilibre au niveau superieur. Rendu operationnel sur un univers fini, le franchissement devient mesurable : *a quel prix un agent accepte-t-il de deplacer son propre jeu, et que devient cet arbitrage quand les deux joueurs le jouent simultanement ?*

> **Fusion** : ce notebook absorbe l'ex-GameTheory-03f-Parcours-Complet — la partie « parcours complet » (sections 4-8) ci-dessous. L'arc méta-actions (marches 1-3) puis l'intégration (chemin, murs, coûts) se lisent d'un seul tenant.


## 0. Le substrat, herite de GT-3b

Un jeu = **deux tables de rangs**, une par joueur, chacune un 4-uplet ordonnant strictement les quatre cases (haut-gauche, haut-droite, bas-gauche, bas-droite) -- rang 4 = meilleur. Les **swaps** `R1, R2, R3` (cote Ligne) et `C1, C2, C3` (cote Colonne) echangent les cases portant deux rangs adjacents `k` et `k+1` : ce sont les six generateurs de l'espace, chacun traverse un mur du monde de Bruns-Kimmich. Toute la mecanique est reprise tel quel de [GT-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb), sans modification -- ce notebook est un consommateur du substrat, pas une refonte.

In [2]:
use std::collections::BTreeSet;

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

// Substrat GT-3b, repris tel quel -- pur stdlib

let mut set02: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set02.insert((a, b, c, d)); }
} } } }
let chambres: Vec<Tab> = set02.into_iter().collect();
let mut jeux: Vec<Jeu> = Vec::new();
for &r in &chambres { for &c in &chambres { jeux.push((r, c)); } }
println!("Chambres strictes par joueur : {} | jeux 2x2 stricts : {}", chambres.len(), jeux.len());
println!("Generateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)");
let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
println!("PD : Ligne {} Colonne {} (rang 4 = meilleur, cases TG TD BG BD)", py_tup4(pd.0), py_tup4(pd.1));


Chambres strictes par joueur : 24 | jeux 2x2 stricts : 576


Generateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)


PD : Ligne (3, 1, 4, 2) Colonne (3, 4, 1, 2) (rang 4 = meilleur, cases TG TD BG BD)


## 1. Marche 1 -- l'agent joue

Avant de changer les regles, il faut les subir. Pour chaque jeu, deux instruments :

- les **equilibres de Nash purs** : cases ou personne ne prefere devier unilateralement ;
- la **dynamique de meilleure reponse** depuis la case haut-gauche : Ligne ajuste, puis Colonne, et on repete -- c'est la facon la plus simple dont des agents "jouent" quand personne ne calcule d'equilibre.

Sur des rangs *ordinaux*, l'equilibre mixte n'a pas de sens (on ne peut pas moyenner des rangs) : un jeu sans equilibre pur est ici **injouable** -- la dynamique cycle sans fin. C'est un fait, pas une anomalie a corriger.

In [3]:
use std::collections::{BTreeMap, BTreeSet};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn py_tup2(t: (usize, usize)) -> String {
    format!("({}, {})", t.0, t.1)
}

fn py_list_tup2(v: &[(usize, usize)]) -> String {
    let items: Vec<String> = v.iter().map(|&t| py_tup2(t)).collect();
    format!("[{}]", items.join(", "))
}

fn py_map_usize(m: &BTreeMap<usize, usize>) -> String {
    let items: Vec<String> = m.iter().map(|(&k, &v)| format!("{}: {}", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}

fn py_map_tup_str(m: &BTreeMap<(usize, &'static str), usize>) -> String {
    let items: Vec<String> = m.iter().map(|(&(k, s), &v)| format!("({}, '{}'): {}", k, s, v)).collect();
    format!("{{{}}}", items.join(", "))
}

fn ne_purs(row: Tab, col: Tab) -> Vec<(usize, usize)> {
    let mut nes = Vec::new();
    for r in 0..2usize {
        for c in 0..2usize {
            if at(row, r, c) > at(row, 1 - r, c) && at(col, r, c) > at(col, r, 1 - c) {
                nes.push((r, c));
            }
        }
    }
    nes
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

let mut set02: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set02.insert((a, b, c, d)); }
} } } }
let chambres: Vec<Tab> = set02.into_iter().collect();
let mut jeux: Vec<Jeu> = Vec::new();
for &r in &chambres { for &c in &chambres { jeux.push((r, c)); } }

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));

// Marche 1 : NE purs + dynamique BR depuis la case haut-gauche

let mut dist_ne: BTreeMap<usize, usize> = BTreeMap::new();
for &(r, c) in &jeux { *dist_ne.entry(ne_purs(r, c).len()).or_insert(0) += 1; }
let mut stats: BTreeMap<&'static str, usize> = BTreeMap::new();
for &(r, c) in &jeux { *stats.entry(br_dyn(r, c).0).or_insert(0) += 1; }
println!("NE purs par jeu : {} sur {}", py_map_usize(&dist_ne), jeux.len());
println!("Dynamique BR depuis TG : converge {} | cycle {}", stats["ok"], stats["cycle"]);
let mut croisement: BTreeMap<(usize, &'static str), usize> = BTreeMap::new();
for &(r, c) in &jeux {
    let key = (ne_purs(r, c).len(), br_dyn(r, c).0);
    *croisement.entry(key).or_insert(0) += 1;
}
println!("Croisement (#NE, BR) : {}", py_map_tup_str(&croisement));
println!();
let nes_pd04 = ne_purs(pd.0, pd.1);
println!("PD : NE purs = {} -> (bas,droite) = defection mutuelle, rang {} chacun", py_list_tup2(&nes_pd04), tab_get(pd.0, 3));


NE purs par jeu : {0: 72, 1: 432, 2: 72} sur 576


Dynamique BR depuis TG : converge 504 | cycle 72


Croisement (#NE, BR) : {(0, 'cycle'): 72, (1, 'ok'): 432, (2, 'ok'): 72}


PD : NE purs = [(1, 1)] -> (bas,droite) = defection mutuelle, rang 2 chacun


### Lecture de la marche 1

La distribution est remarquablement symetrique : **72 jeux sans equilibre pur, 432 avec un seul, 72 avec deux** (les coordinations). Et le croisement est exact : la dynamique converge **si et seulement si** un equilibre pur existe -- les 72 cycles sont precisement les 72 jeux sans equilibre. Aucun jeu a deux equilibres ne pose ici de probleme de selection *dynamique* : parti de haut-gauche, le jeu se stabilise toujours quelque part.

Le Dilemme du Prisonnier fait ce qu'on attend de lui : equilibre unique a la defection mutuelle, rang 2 chacun -- le rang 4 de la cooperation existe mais n'est pas stable. Retenons ce chiffre, **2** : c'est ce que vaut le Dilemme pour celui qui y est piege.

In [4]:
type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

// Le fait brut : exhiber un jeu injouable (cycle BR complet)

let ex_injouable: Jeu = ((2, 3, 1, 4), (1, 2, 4, 3));
let (row, col) = ex_injouable;
println!("Jeu sans NE pur : Ligne {} Colonne {}", py_tup4(row), py_tup4(col));
let noms_cases = ["TG", "TD", "BG", "BD"];
let (mut r, mut c) = (0usize, 0usize);
let mut trace = vec![(r, c)];
for _ in 0..6 {
    let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
    if nr != r { r = nr; trace.push((r, c)); }
    let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
    if nc != c { c = nc; trace.push((r, c)); }
}
let s1: Vec<String> = trace.iter().take(6).map(|&(tr, tc)| noms_cases[2 * tr + tc].to_string()).collect();
println!("Trajectoire BR depuis TG : {} -> ... (cycle)", s1.join(" -> "));
let s2: Vec<String> = trace.iter().skip(1).take(4).map(|&(tr, tc)| noms_cases[2 * tr + tc].to_string()).collect();
println!("Boucle : {} -> ... repetee indefiniment", s2.join(" -> "));
println!();
println!("Convention : gain au sorti de BR ; jeu injouable -> 0 pour chacun (pire que tout rang >= 1)");


Jeu sans NE pur : Ligne (2, 3, 1, 4) Colonne (1, 2, 4, 3)


Trajectoire BR depuis TG : TG -> TD -> BD -> BG -> TG -> TD -> ... (cycle)


Boucle : TD -> BD -> BG -> TG -> ... repetee indefiniment


Convention : gain au sorti de BR ; jeu injouable -> 0 pour chacun (pire que tout rang >= 1)


## 2. Marche 2 -- l'agent deplace le jeu, et paie

L'agent Ligne recoit desormais des **meta-actions** : appliquer l'un de ses trois swaps `R1, R2, R3` -- reecrire ses propres preference declarees -- puis jouer le jeu deplace. Chaque swap coute **c echelons de rang**. Cette unite n'est pas un artifice : payer 1 signifie *renoncer a un echelon de preference pour l'atteindre*, ce qui rend la soustraction `rang final - cout` honnetement interpretable dans un monde purement ordinal.

Ligne arbitre : rester et toucher son rang courant, ou migrer vers une table distante de d swaps et toucher `rang(final) - c*d`. Comme l'espace est fini, l'optimum se calcule par parcours exhaustif -- BFS depuis **sa propre table** (le graphe de Cayley des 24 tables, diametre 6, deja mesure par GT-3b).

In [5]:
use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

let mut set02: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set02.insert((a, b, c, d)); }
} } } }
let chambres: Vec<Tab> = set02.into_iter().collect();
let mut jeux: Vec<Jeu> = Vec::new();
for &r in &chambres { for &c in &chambres { jeux.push((r, c)); } }

// Marche 2 : optimum de migration par jeu, puis sweep du cout

let mut best_by_dist: BTreeMap<Jeu, BTreeMap<usize, usize>> = BTreeMap::new();
for &(row_t, col_t) in &jeux {
    let mut m: BTreeMap<usize, usize> = BTreeMap::new();
    for (&t2, &d) in &bfs_from(row_t) {
        let p = payoff(t2, col_t, 0);
        m.entry(d).and_modify(|e| *e = (*e).max(p)).or_insert(p);
    }
    best_by_dist.insert((row_t, col_t), m);
}
println!("c   | %migrer | dist.opt moy | gain net moy (migrants) | fuient un cycle");
println!("{}", "-".repeat(72));
let n = jeux.len();
for c in 0..4usize {
    let (mut n_mig, mut tot_d, mut tot_gain, mut fuit) = (0usize, 0usize, 0i64, 0usize);
    for (&(r_t, c_t), m) in &best_by_dist {
        let mut best = i64::MIN;
        for (&d, &v) in m { best = best.max(v as i64 - c as i64 * d as i64); }
        let mut bd = usize::MAX;
        for (&d, &v) in m { if v as i64 - c as i64 * d as i64 == best { bd = bd.min(d); } }
        if bd > 0 && best > m[&0] as i64 {
            n_mig += 1;
            tot_d += bd;
            tot_gain += best - m[&0] as i64;
            if payoff(r_t, c_t, 0) == 0 { fuit += 1; }
        }
    }
    println!("{}   |  {:>3}%   |     {:.2}     |          {:+.2}          |     {}",
        c, 100 * n_mig / n,
        tot_d as f64 / n_mig.max(1) as f64,
        tot_gain as f64 / n_mig.max(1) as f64,
        fuit);
};


c   | %migrer | dist.opt moy | gain net moy (migrants) | fuient un cycle


------------------------------------------------------------------------


0   |   56%   |     1.33     |          +1.93          |     72


1   |   16%   |     1.25     |          +2.00          |     72


2   |    8%   |     1.00     |          +1.50          |     48


3   |    4%   |     1.00     |          +1.00          |     24


### Lecture du sweep : un seuil monotone, une distance qui se contracte

Quatre faits mesures :

1. **La migration s'effondre avec le cout** : 56 % des jeux voient Ligne migrer a cout nul, 16 % a cout 1, 8 % a cout 2, 4 % a cout 3. A un echelon par swap, six jeux sur sept restent -- le privilege de reecrire ses preferences est cher.
2. **La distance optimale se contracte** de 1,33 vers 1,00 : plus le km de swap est cher, plus on ne paie que le deplacement minimal -- a cout eleve, seuls les gains de proximite immediate survivent.
3. **Le gain net moyen des migrants monte puis redescend** (+1,93 a c=0, +2,00 a c=1, +1,50 a c=2, +1,00 a c=3) : a c=0 partent aussi les migrations gadgets ; a c=1 ne restent que les migrations de fond ; au-dela, le cout ronge le gain.
4. **Les 72 jeux injouables se vident a leur rythme** : meme a cout 1, les 72 fuient tous (quitter un gain nul vaut n'importe quel prix raisonnable) ; a cout 2 il en reste 48, a cout 3 seulement 24 -- les autres preferent le cycle gratuit au deplacement trop cher. *Meme l'injouable a un prix au-dessus duquel on le tolere.*

In [6]:
use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

let mut set02: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set02.insert((a, b, c, d)); }
} } } }
let chambres: Vec<Tab> = set02.into_iter().collect();
let mut jeux: Vec<Jeu> = Vec::new();
for &r in &chambres { for &c in &chambres { jeux.push((r, c)); } }

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));

let mut best_by_dist: BTreeMap<Jeu, BTreeMap<usize, usize>> = BTreeMap::new();
for &(row_t, col_t) in &jeux {
    let mut m: BTreeMap<usize, usize> = BTreeMap::new();
    for (&t2, &d) in &bfs_from(row_t) {
        let p = payoff(t2, col_t, 0);
        m.entry(d).and_modify(|e| *e = (*e).max(p)).or_insert(p);
    }
    best_by_dist.insert((row_t, col_t), m);
}

// Le Dilemme sous les meta-actions : que peut acheter Ligne ?

let m_pd = best_by_dist[&pd].clone();
println!("PD : rang courant de Ligne (defection mutuelle) = {}", m_pd[&0]);
println!("Meilleur rang atteignable par distance :");
for (&d, &v) in &m_pd {
    let extra = if d > 0 { format!("  -> net a c=1 : {}", v as i64 - d as i64) } else { "   (rester)".to_string() };
    println!("  a distance {} : rang {}{}", d, v, extra);
}
println!();
for c in 0..3usize {
    let mut best = i64::MIN;
    for (&d, &v) in &m_pd { best = best.max(v as i64 - c as i64 * d as i64); }
    let mut bd = usize::MAX;
    for (&d, &v) in &m_pd { if v as i64 - c as i64 * d as i64 == best { bd = bd.min(d); } }
    let verdict = if bd == 0 { "RESTE".to_string() } else { format!("MIGRE a distance {}", bd) };
    println!("c = {} : net optimal {} (courant {}) -> Ligne {}", c, best, m_pd[&0], verdict);
};


PD : rang courant de Ligne (defection mutuelle) = 2


Meilleur rang atteignable par distance :


  a distance 0 : rang 2   (rester)


  a distance 1 : rang 3  -> net a c=1 : 2


  a distance 2 : rang 4  -> net a c=1 : 2


  a distance 3 : rang 4  -> net a c=1 : 1


... (2 outputs left out)

  a distance 6 : rang 4  -> net a c=1 : -2


c = 0 : net optimal 4 (courant 2) -> Ligne MIGRE a distance 2


c = 1 : net optimal 2 (courant 2) -> Ligne RESTE


c = 2 : net optimal 2 (courant 2) -> Ligne RESTE


### Lecture : le Dilemme exactement indifferent

A cout nul, Ligne quitte le Dilemme : deux swaps l'amennent a un jeu ou sa table lui donne le rang 4, net +2. Mais **a cout 1, l'indifference est exacte** : le rang 3 a distance 1 et le rang 4 a distance 2 donnent tous deux un net de 2 -- strictement egal a rester. Le gain de fuite du Dilemme est **entierement mange par le cout du deplacement**. Ce n'est pas un accident d'arrondi : sur l'echelle des rangs, la fuite solitaire du Dilemme coute precisement ce qu'elle rapporte. Le piege de la strate 6 a une propriete economique de plus : il est *juste assez solide* pour retenir un agent solitaire qui doit payer son deplacement -- ce qui rend d'autant plus interessante la question des deux agents, en marche 3.

In [7]:
use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn ne_purs(row: Tab, col: Tab) -> Vec<(usize, usize)> {
    let mut nes = Vec::new();
    for r in 0..2usize {
        for c in 0..2usize {
            if at(row, r, c) > at(row, 1 - r, c) && at(col, r, c) > at(col, r, 1 - c) {
                nes.push((r, c));
            }
        }
    }
    nes
}

fn py_jeu(j: Jeu) -> String {
    format!("({}, {})", py_tup4(j.0), py_tup4(j.1))
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

fn py_list_tup4(v: &[Tab]) -> String {
    let items: Vec<String> = v.iter().map(|&t| py_tup4(t)).collect();
    format!("[{}]", items.join(", "))
}

let mut set02: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set02.insert((a, b, c, d)); }
} } } }
let chambres: Vec<Tab> = set02.into_iter().collect();
let mut jeux: Vec<Jeu> = Vec::new();
for &r in &chambres { for &c in &chambres { jeux.push((r, c)); } }

let mut best_by_dist: BTreeMap<Jeu, BTreeMap<usize, usize>> = BTreeMap::new();
for &(row_t, col_t) in &jeux {
    let mut m: BTreeMap<usize, usize> = BTreeMap::new();
    for (&t2, &d) in &bfs_from(row_t) {
        let p = payoff(t2, col_t, 0);
        m.entry(d).and_modify(|e| *e = (*e).max(p)).or_insert(p);
    }
    best_by_dist.insert((row_t, col_t), m);
}

// Les migrants a cout 1 : qui part, et la fuite des injouables

let mut mig: Vec<(Tab, Tab, usize, usize, usize)> = Vec::new();
for (&(r_t, c_t), m) in &best_by_dist {
    let mut best = i64::MIN;
    for (&d, &v) in m { best = best.max(v as i64 - d as i64); }
    let mut bd = usize::MAX;
    for (&d, &v) in m { if v as i64 - d as i64 == best { bd = bd.min(d); } }
    if bd > 0 && best > m[&0] as i64 {
        mig.push((r_t, c_t, bd, m[&0], m[&bd]));
    }
}
let n_ref = mig.iter().filter(|x| x.3 == 0).count();
println!("Migrants a c=1 : {} jeux sur {} dont {} refugies (partent d'un rang nul)", mig.len(), jeux.len(), n_ref);
let opp: Vec<(Tab, Tab, usize, usize, usize)> = mig.iter().filter(|x| x.3 > 0).copied().collect();
let (r_t, c_t, bd, p0, p1) = opp[0];
let mut t_cible_opt = None;
for (&t2, &d) in &bfs_from(r_t) {
    if d == bd && payoff(t2, c_t, 0) == p1 { t_cible_opt = Some(t2); break; }
}
let t_cible = t_cible_opt.unwrap();
println!("Opportuniste : jeu {} | rang {} -> rang {} en {} swap (table cible {} )", py_jeu((r_t, c_t)), p0, p1, bd, py_tup4(t_cible));
let inj: Vec<Jeu> = jeux.iter().copied().filter(|&g| ne_purs(g.0, g.1).is_empty()).collect();
let rep1 = inj.iter().filter(|&&g| {
    let m = &best_by_dist[&g];
    let mut mind = usize::MAX;
    for (&d, &v) in m { if v > 0 { mind = mind.min(d); } }
    mind == 1
}).count();
println!();
println!("Jeux injouables : {} | reparables en 1 swap : {} | autres : {}", inj.len(), rep1, inj.len() - rep1);
let (row_i, col_i) = inj[0];
let jouables_d1: Vec<Tab> = bfs_from(row_i).iter()
    .filter(|&(&t2, &d)| d == 1 && payoff(t2, col_i, 0) > 0)
    .map(|(&t2, _)| t2)
    .collect();
let meilleur: Option<usize> = jouables_d1.iter().map(|&t2| payoff(t2, col_i, 0)).max();
let meilleur_disp = match meilleur { Some(v) => v.to_string(), None => "None".to_string() };
println!("Exemple {} -> tables jouables a distance 1 : {} | meilleur rang {}",
    py_jeu((row_i, col_i)), py_list_tup4(&jouables_d1), meilleur_disp);


Migrants a c=1 : 96 jeux sur 576 dont 72 refugies (partent d'un rang nul)


Opportuniste : jeu ((1, 4, 2, 3), (1, 2, 4, 3)) | rang 2 -> rang 4 en 1 swap (table cible (2, 4, 1, 3) )


Jeux injouables : 72 | reparables en 1 swap : 48 | autres : 24


Exemple ((1, 3, 4, 2), (2, 1, 3, 4)) -> tables jouables a distance 1 : [(1, 2, 4, 3)] | meilleur rang 3


### Lecture : deux raisons de partir

Les migrants a cout 1 ne sont pas une population homogene. Il y a les **opportunistes** -- des jeux jouables ou un seul swap proche decalle l'equilibre vers une case de meilleur rang (l'exemple affiche saute de deux echelons en un swap) -- et les **refugies** : les 72 jeux sans equilibre, dont 48 se reparent a distance 1. Pour un refugie, la meta-action n'est pas du luxe strategique, c'est la condition meme de jouabilite : sans elle, le jeu ne produit aucun resultat -- c'est le refugie affiche plus bas, parti du rang nul. La strate 7 commence deja ici -- *changer les regles pour que le jeu existe* -- avant meme toute consideration d'amelioration.

### Le mur traversé — ce que le migrant coupe (critère de clôture #12207)

Le chantier #12207 se clôt quand un lecteur peut, dans un notebook exécuté : partir d'un jeu nommé, atteindre un autre jeu nommé par un chemin qu'il n'a pas écrit lui-même, **voir le mur qu'il traverse**, et lire le coût de la méta-action qui l'y a mené. Les marches 1-2 ont nommé les jeux, calculé les chemins optimaux et balayé les coûts — mais aucune sortie ne **montrait** le mur. Entre deux chambres strictes adjacentes, l'égalité des deux cases permutées est un mur de codimension 1 : la lecture Bruns-Kimmich de GT-3b, rendue visible ici dans le flot de migration payée.

In [8]:
type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn table_depuis_u(u: [f64; 4]) -> Tab {
    let mut ordre: Vec<usize> = vec![0, 1, 2, 3];
    ordre.sort_by(|&a, &b| u[a].partial_cmp(&u[b]).unwrap());
    let mut t = [0usize; 4];
    for (rang, &j) in ordre.iter().enumerate() {
        t[j] = rang + 1;
    }
    (t[0], t[1], t[2], t[3])
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

// Le mur traverse : la meta-action echange les rangs k et k+1 de deux cases. Dans l'espace
// des utilites (representant lineaire u = valeur du rang), la deformation
// u(lambda) = (1-lambda)*u_depart + lambda*u_cible coupe l'hyperplan d'egalite des deux
// cases permutees. De part et d'autre, la table ordinale est CONSTANTE : c'est la chambre.

let (r_t, t_cible, bd, p0, p1) = ((1usize, 4, 2, 3), (2usize, 4, 1, 3), 1usize, 2usize, 4usize);
let positions: Vec<usize> = (0..4).filter(|&i| tab_get(r_t, i) != tab_get(t_cible, i)).collect();
assert_eq!(positions.len(), 2, "une meta-action = un swap de deux cases");
let (ia, ib) = (positions[0], positions[1]);
let noms = ["TG", "TD", "BG", "BD"];
println!("Jeu depart (Ligne) : {}  ->  cible : {}  ({} meta-action, rang {} -> {})", py_tup4(r_t), py_tup4(t_cible), bd, p0, p1);
println!("Cases permutees : {} (rang {}) <-> {} (rang {})", noms[ia], tab_get(r_t, ia), noms[ib], tab_get(r_t, ib));
println!();
println!("lambda |   u_TG    u_TD    u_BG    u_BD  | table ordinale induite");
println!("{}", "-".repeat(66));
for lam in [0.0f64, 0.25, 0.5, 0.75, 1.0] {
    let u: [f64; 4] = std::array::from_fn(|j| (1.0 - lam) * tab_get(r_t, j) as f64 + lam * tab_get(t_cible, j) as f64);
    let classe = if (u[ia] - u[ib]).abs() < 1e-12 {
        format!("MUR : u_{} = u_{} -- codimension 1", noms[ia], noms[ib])
    } else {
        let t = table_depuis_u(u);
        if t == r_t { "chambre de depart".to_string() } else if t == t_cible { "chambre cible".to_string() } else { py_tup4(t) }
    };
    let cells: Vec<String> = u.iter().map(|x| format!("{:5.2}", x)).collect();
    println!("  {:.2} | {} | {}", lam, cells.join("  "), classe);
}
println!();
let m15 = tab_get(r_t, ia).min(tab_get(r_t, ib));
println!("Le migrant coupe UN mur (l'egalite {}/{} des rangs {} et {})", noms[ia], noms[ib], m15, m15 + 1);
println!("a lambda = 0.50, et l'a paye c = {} : rang {} -> rang {}, gain net {:+}.", bd, p0, p1, p1 as i64 - p0 as i64 - bd as i64);


Jeu depart (Ligne) : (1, 4, 2, 3)  ->  cible : (2, 4, 1, 3)  (1 meta-action, rang 2 -> 4)


Cases permutees : TG (rang 1) <-> BG (rang 2)


lambda |   u_TG    u_TD    u_BG    u_BD  | table ordinale induite


------------------------------------------------------------------


  0.00 |  1.00   4.00   2.00   3.00 | chambre de depart


  0.25 |  1.25   4.00   1.75   3.00 | chambre de depart


  0.50 |  1.50   4.00   1.50   3.00 | MUR : u_TG = u_BG -- codimension 1


  0.75 |  1.75   4.00   1.25   3.00 | chambre cible


  1.00 |  2.00   4.00   1.00   3.00 | chambre cible


Le migrant coupe UN mur (l'egalite TG/BG des rangs 1 et 2)


a lambda = 0.50, et l'a paye c = 1 : rang 2 -> rang 4, gain net +1.


### Lecture : les quatre maillons enchaînés

La sortie ci-dessus est le critère de clôture de #12207 réalisé en un seul flot exécuté : un **jeu nommé** (le premier migrant opportuniste, tables affichées), rejoint par un **chemin qu'il n'a pas écrit** (le BFS optimal de la marche 2), en **traversant un mur visible** — la table ordinale reste celle de la chambre de départ pour tout λ < 0.5, celle de la chambre cible pour tout λ > 0.5, et l'égalité des deux cases permutées à λ = 0.50 exactement est le mur de codimension 1 — et le **coût lu** : c = 1 pour cette traversée, gain net +1 en rang.

Le passage payé se lit désormais `chambre -> mur -> chambre voisine` : la méta-action tarifée achète exactement une traversée de mur, et rien d'autre — un échange de rangs qui ne modifie l'ordre d'aucune autre case ne peut couper plus d'un mur.

## 3. Marche 3 -- deux agents, un meta-jeu

Derniere marche : **Ligne et Colonne disposent simultanement de leurs meta-actions**. Chacun choisit une action dans {rester, R1, R2, R3} x {rester, C1, C2, C3} -- chacun ne reecrit que **sa** table, payer ses propres echelons -- puis le jeu deplace est joue (dynamique BR depuis haut-gauche, convention de marche 1), et chacun touche son rang final moins son cout. Seize profils par jeu de base : un **meta-jeu 4x4** dont on peut chercher les equilibres de Nash purs, exactement comme a la marche 1.

La question du desaccord devient calculable : *existe-t-il un equilibre du meta-jeu, bouge-t-on a l'equilibre, et cet equilibre est-il bon pour les deux ?*

In [9]:
use std::collections::{BTreeMap, BTreeSet};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn meta_profil(row_t: Tab, col_t: Tab, a1: usize, a2: usize, cout: i64) -> (i64, i64) {
    let r2 = if a1 > 0 { swap_val(row_t, a1) } else { row_t };
    let c2 = if a2 > 0 { swap_val(col_t, a2) } else { col_t };
    let p1 = payoff(r2, c2, 0) as i64;
    let p2 = payoff(r2, c2, 1) as i64;
    (p1 - if a1 > 0 { cout } else { 0 }, p2 - if a2 > 0 { cout } else { 0 })
}

fn meta_ne(row_t: Tab, col_t: Tab, cout: i64) -> (Vec<(usize, usize)>, BTreeMap<(usize, usize), (i64, i64)>) {
    let mut profils = BTreeMap::new();
    for a1 in 0..4usize {
        for a2 in 0..4usize {
            profils.insert((a1, a2), meta_profil(row_t, col_t, a1, a2, cout));
        }
    }
    let mut nes = Vec::new();
    for (&(a1, a2), &(u1, u2)) in &profils {
        let ok1 = (0..4usize).all(|a1b| profils[&(a1b, a2)].0 <= u1);
        let ok2 = (0..4usize).all(|a2b| profils[&(a1, a2b)].1 <= u2);
        if ok1 && ok2 { nes.push((a1, a2)); }
    }
    (nes, profils)
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn py_map_usize(m: &BTreeMap<usize, usize>) -> String {
    let items: Vec<String> = m.iter().map(|(&k, &v)| format!("{}: {}", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}

let mut set02: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set02.insert((a, b, c, d)); }
} } } }
let chambres: Vec<Tab> = set02.into_iter().collect();
let mut jeux: Vec<Jeu> = Vec::new();
for &r in &chambres { for &c in &chambres { jeux.push((r, c)); } }

// Marche 3 : le meta-jeu 4x4 et ses equilibres (cout = 1)

let mut st_ne: BTreeMap<usize, usize> = BTreeMap::new();
let (mut tous_restant, mut ambigu, mut aucun_restant) = (0usize, 0usize, 0usize);
for &(row_t, col_t) in &jeux {
    let (nes, _) = meta_ne(row_t, col_t, 1);
    *st_ne.entry(nes.len()).or_insert(0) += 1;
    if nes.is_empty() { continue; }
    if nes.iter().all(|&n| n == (0, 0)) { tous_restant += 1; }
    else if nes.contains(&(0, 0)) { ambigu += 1; }
    else { aucun_restant += 1; }
}
let n_meta = st_ne.iter().filter(|&(k, _)| *k > 0).map(|(_, v)| v).sum::<usize>();
println!("Jeux avec au moins un meta-NE pur : {} / {}", n_meta, jeux.len());
println!("Distribution du nombre de meta-NE : {}", py_map_usize(&st_ne));
println!();
println!("Statut du mouvement a l'equilibre :");
println!("  tous les meta-NE restent sur place : {}", tous_restant);
println!("  (rester) coexiste avec des NE mobiles : {} (selection d'equilibre)", ambigu);
println!("  aucun meta-NE ne reste sur place : {} (bouger est necessaire)", aucun_restant);


Jeux avec au moins un meta-NE pur : 572 / 576


Distribution du nombre de meta-NE : {0: 4, 1: 166, 2: 266, 3: 16, 4: 116, 5: 6, 6: 2}


Statut du mouvement a l'equilibre :


  tous les meta-NE restent sur place : 134


  (rester) coexiste avec des NE mobiles : 332 (selection d'equilibre)


  aucun meta-NE ne reste sur place : 106 (bouger est necessaire)


### Lecture : le meta-jeu a presque toujours des equilibres

Premiere surprise : **572 jeux sur 576** ont au moins un equilibre pur au niveau meta -- ajouter des meta-actions *stabilise* plutot qu'elle ne destabilise. Le detail du statut est plus riche :

- **134 jeux** : tous les equilibres restent sur place -- la meta-action existe mais personne n'a interet a s'en servir a l'equilibre (l'immobilisme est stable) ;
- **332 jeux** : rester et bouger coexistent comme equilibres -- c'est le domaine de la *selection d'equilibre* : plusieurs futurs possibles, aucun critere ordinal n'en designe un ;
- **106 jeux** : aucun equilibre sur place -- **bouger est une necessite d'equilibre**. Dans pres d'un jeu sur cinq, l'equilibre de strate 6 n'est plus tenable des que reecrire les regles est possible : l'agent qui refuse de payer reste piege pendant que l'autre reconfigure.

La marche 2 disait *quand un agent solitaire part* ; la marche 3 dit que **des qu'on ouvre la porte aux deux, dans un cinquieme des jeux, partir est force**.

In [10]:
use std::collections::BTreeMap;

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn meta_profil(row_t: Tab, col_t: Tab, a1: usize, a2: usize, cout: i64) -> (i64, i64) {
    let r2 = if a1 > 0 { swap_val(row_t, a1) } else { row_t };
    let c2 = if a2 > 0 { swap_val(col_t, a2) } else { col_t };
    let p1 = payoff(r2, c2, 0) as i64;
    let p2 = payoff(r2, c2, 1) as i64;
    (p1 - if a1 > 0 { cout } else { 0 }, p2 - if a2 > 0 { cout } else { 0 })
}

fn meta_ne(row_t: Tab, col_t: Tab, cout: i64) -> (Vec<(usize, usize)>, BTreeMap<(usize, usize), (i64, i64)>) {
    let mut profils = BTreeMap::new();
    for a1 in 0..4usize {
        for a2 in 0..4usize {
            profils.insert((a1, a2), meta_profil(row_t, col_t, a1, a2, cout));
        }
    }
    let mut nes = Vec::new();
    for (&(a1, a2), &(u1, u2)) in &profils {
        let ok1 = (0..4usize).all(|a1b| profils[&(a1b, a2)].0 <= u1);
        let ok2 = (0..4usize).all(|a2b| profils[&(a1, a2b)].1 <= u2);
        if ok1 && ok2 { nes.push((a1, a2)); }
    }
    (nes, profils)
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn py_tup2(t: (usize, usize)) -> String {
    format!("({}, {})", t.0, t.1)
}

fn py_list_tup2(v: &[(usize, usize)]) -> String {
    let items: Vec<String> = v.iter().map(|&t| py_tup2(t)).collect();
    format!("[{}]", items.join(", "))
}

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));

// Le meta-jeu du Dilemme, en entier

let (nes_pd, prof_pd) = meta_ne(pd.0, pd.1, 1);
let noms20 = ["reste", "R1", "R2", "R3"];
let noms_c = ["reste", "C1", "C2", "C3"];
println!("Meta-jeu du PD (Ligne en lignes, Colonne en colonnes) :");
let head: Vec<String> = noms_c.iter().map(|n| format!("{:>10}", n)).collect();
println!("          {}", head.join(""));
for a1 in 0..4usize {
    let mut ligne = format!("{:>8}  ", noms20[a1]);
    for a2 in 0..4usize {
        let (u1, u2) = prof_pd[&(a1, a2)];
        let marque = if nes_pd.contains(&(a1, a2)) { " *" } else { "  " };
        ligne += &format!("({},{}){} ", u1, u2, marque);
    }
    println!("{}", ligne);
}
println!();
println!("* = equilibre de Nash pur du meta-jeu : {}", py_list_tup2(&nes_pd));


Meta-jeu du PD (Ligne en lignes, Colonne en colonnes) :


               reste        C1        C2        C3


   reste  (2,2) * (4,1)   (2,2) * (2,1)   


      R1  (1,4)   (3,1)   (1,3)   (-1,-1)   


      R2  (2,2) * (3,1)   (2,2) * (2,1)   


      R3  (1,2)   (-1,-1)   (1,2)   (3,3) * 


* = equilibre de Nash pur du meta-jeu : [(0, 0), (0, 2), (2, 0), (2, 2), (3, 3)]


### Lecture : le Dilemme se resout -- a deux, et par equilibre

Le meta-jeu du Dilemme porte **cinq equilibres**. Quatre laissent les deux prisonniers a (2,2) -- le statu quo auquel on pense toujours. Mais le cinquieme, **(R3, C3), paie (3,3)** : chacun reecrit sa propre preference au niveau 3-4, le jeu deplace devient une coordination dont l'equilibre selectionne par la dynamique est la case mutuellement meilleure (rang 4 chacun), et meme apres avoir paye un echelon chacun, il reste **(3,3) > (2,2)**. Trois proprietes remarquables :

1. **L'evasion est unilateralement stable** : si un seul migre, la dynamique retombe sur la defection mutuelle et le migrant a paye pour rien -- les profils (R3, reste) et (reste, C3) du tableau paient (1, 2) et (2, 1) : moins que (3, 3) pour chacun -- personne ne peut profiter du départ de l'autre ;
2. **Elle est symetrique** : chacun ne touche qu'a sa table ; personne ne reecrit les preferences de l'autre ;
3. **Elle n'est pas unique** : quatre autres equilibres restent au (2,2). L'evasion *existe et est stable*, mais rien ne la *selectionne* -- c'est un probleme de focalisation, plus de stabilite.

C'est la reponse mesuree a la question de la marche : *modifier les regles est une action, et dans le Dilemme cette action est un equilibre qui domine le piege -- a condition que les deux la jouent ensemble.*

In [11]:
use std::collections::{BTreeMap, BTreeSet};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn meta_profil(row_t: Tab, col_t: Tab, a1: usize, a2: usize, cout: i64) -> (i64, i64) {
    let r2 = if a1 > 0 { swap_val(row_t, a1) } else { row_t };
    let c2 = if a2 > 0 { swap_val(col_t, a2) } else { col_t };
    let p1 = payoff(r2, c2, 0) as i64;
    let p2 = payoff(r2, c2, 1) as i64;
    (p1 - if a1 > 0 { cout } else { 0 }, p2 - if a2 > 0 { cout } else { 0 })
}

fn meta_ne(row_t: Tab, col_t: Tab, cout: i64) -> (Vec<(usize, usize)>, BTreeMap<(usize, usize), (i64, i64)>) {
    let mut profils = BTreeMap::new();
    for a1 in 0..4usize {
        for a2 in 0..4usize {
            profils.insert((a1, a2), meta_profil(row_t, col_t, a1, a2, cout));
        }
    }
    let mut nes = Vec::new();
    for (&(a1, a2), &(u1, u2)) in &profils {
        let ok1 = (0..4usize).all(|a1b| profils[&(a1b, a2)].0 <= u1);
        let ok2 = (0..4usize).all(|a2b| profils[&(a1, a2b)].1 <= u2);
        if ok1 && ok2 { nes.push((a1, a2)); }
    }
    (nes, profils)
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn pareto_dominated(profils: &BTreeMap<(usize, usize), (i64, i64)>, p: (i64, i64)) -> bool {
    profils.values().any(|v| v.0 > p.0 && v.1 > p.1)
}

fn py_jeu(j: Jeu) -> String {
    format!("({}, {})", py_tup4(j.0), py_tup4(j.1))
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

fn py_tup2i(t: (i64, i64)) -> String {
    format!("({}, {})", t.0, t.1)
}

fn py_list_tup2i(v: &[(i64, i64)]) -> String {
    let items: Vec<String> = v.iter().map(|&t| py_tup2i(t)).collect();
    format!("[{}]", items.join(", "))
}

let mut set02: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set02.insert((a, b, c, d)); }
} } } }
let chambres: Vec<Tab> = set02.into_iter().collect();
let mut jeux: Vec<Jeu> = Vec::new();
for &r in &chambres { for &c in &chambres { jeux.push((r, c)); } }

// L'echec de coordination : quand AUCUN equilibre n'est Pareto-optimal

let (mut au_moins_un_bon, mut aucun_bon) = (0usize, 0usize);
let mut ex_dur: Option<(Jeu, Vec<(usize, usize)>, BTreeMap<(usize, usize), (i64, i64)>)> = None;
for &(row_t, col_t) in &jeux {
    let (nes, profils) = meta_ne(row_t, col_t, 1);
    if nes.is_empty() { continue; }
    let any_good = nes.iter().any(|&n| !pareto_dominated(&profils, profils[&n]));
    if any_good { au_moins_un_bon += 1; }
    else {
        aucun_bon += 1;
        if ex_dur.is_none() { ex_dur = Some(((row_t, col_t), nes, profils)); }
    }
}
println!("Jeux (avec meta-NE) dont AU MOINS UN equilibre est Pareto-optimal : {}", au_moins_un_bon);
println!("Jeux ou AUCUN equilibre n'est Pareto-optimal : {} -- l'echec de coordination dur", aucun_bon);
let (g_dur, nes_dur, prof_dur) = ex_dur.unwrap();
let paie: Vec<(i64, i64)> = nes_dur.iter().map(|&n| prof_dur[&n]).collect();
println!();
println!("Exemple {} : les meta-NE paient {}", py_jeu(g_dur), py_list_tup2i(&paie));
let frontiere: Vec<(i64, i64)> = prof_dur.iter()
    .filter(|(_, v)| !pareto_dominated(&prof_dur, **v))
    .map(|(_, v)| *v)
    .take(5)
    .collect();
println!("Frontiere de Pareto du meta-jeu : {} ...", py_list_tup2i(&frontiere));


Jeux (avec meta-NE) dont AU MOINS UN equilibre est Pareto-optimal : 568


Jeux ou AUCUN equilibre n'est Pareto-optimal : 4 -- l'echec de coordination dur


Exemple ((1, 3, 2, 4), (3, 4, 2, 1)) : les meta-NE paient [(2, 2), (2, 2), (2, 2), (2, 2)]


Frontiere de Pareto du meta-jeu : [(4, 1), (3, 1), (1, 3), (3, 1), (3, 3)] ...


### Lecture : la limite du decentralise

Dans **568 jeux sur 572**, au moins un equilibre du meta-jeu est Pareto-optimal : en general, le meilleur a deux est stable, le probleme est seulement de *le choisir* parmi les equilibres. Mais **4 jeux realisent l'echec de coordination pur** : tous leurs equilibres paient (2,2) tandis que la frontiere de Pareto contient des profils mutuellement meilleurs -- des regles reecrites dont les deux joueurs profiteraient, et qui ne sont **pas stables** : chacun serait tente d'en devier. L'exhibe le montre chiffres en main : l'accord existe dans la matrice, il n'existe pas dans les incitations.

C'est exactement la jonction annoncee par le chantier : la ou la strate 7 rencontre la theorie des mecanismes. Quand changer les regles est une action decentralisee, l'accord mutuel sur le changement reste un equilibre a produire -- le probleme ne fait que remonter d'un etage. Un protocole d'engagement (le versant commitment de [GT-9b](GameTheory-09b-Commitment-Stackelberg-Python.ipynb)) est le candidat naturel, et l'exercice 3 le fait toucher du doigt.

## Limites et conventions (lues avant d'etendre)

- **Le cout en echelons de rang** est une convention locale, documentee : elle rend `rang - cout` interpretable sans cardinaliser les preferences. Un cout en energie, en temps ou en argent exigerait une echelle cardinale et changerait les seuils -- pas la forme du protocole.
- **La regle de jeu** est la dynamique BR depuis la case haut-gauche. Une autre regle de selection (depuis une autre case, tirage aleatoire, dynamique simultanee) deplace certains equilibres selectionnes ; les comptages d'equilibres *purs* (marche 1) et de meta-NE (marche 3) n'en dependent pas.
- **Le meta-jeu est one-shot et simultane** : pas de repetition, pas d'engagement, pas d'ordre de passage. L'exercice 3 ouvre le sequentiel.
- **Chacun ne reecrit que sa table** : le redesign institutionnel croise (reecrire la table de l'autre, ou une table commune) est hors perimetre -- c'est une strate encore au-dessus.
- Les dettes de verification du chantier (passage 576 vers 144, tore a 37 trous, references arXiv non ouvertes firsthand) restent **RAPPORTEES** et ne sont pas utilisees ici : ce notebook ne derive que du substrat exact de GT-3b, lui-meme exhaustivement verifie.

## 4. Le parcours complet — du jeu nommé au coût de la méta-action

**Versant d'intégration du chantier #12207, absorbé ici en seconde partie.** Les versants précédents ont livré chacun une pièce : [Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) construit et vérifie des chemins minimaux entre chambres, [GT-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) identifie les murs où vivent les égalités, la Partie I ci-dessus (marches 1-3) a tarifé les swaps comme des actions que l'agent paie. Cette seconde partie referme la boucle : il réunit les trois pièces **dans un seul parcours**, celui que le chantier demande de bout en bout --

> un lecteur peut, dans un notebook exécuté : partir d'un **jeu nommé**, atteindre un autre jeu nommé par un **chemin qu'il n'a pas écrit lui-même**, **voir le mur** qu'il traverse à chaque pas, et **lire le coût** de la méta-action qui l'y a mené.

## Plan

1. **Le chemin que le lecteur n'écrit pas** : constructeur BFS et vérificateur indépendant (patron GT-20)
2. **Voir les murs** : chaque pas traverse un jeu à égalité codimension 1, dont on exhibe les deux faces
3. **Le coût des méta-actions** : tarifs par côté puis par niveau -- et un résultat inattendu, mesuré
4. **Le parcours complet, bout en bout** : le bloc intégral, puis sa re-vérification indépendante
5. **Exercices**

## La question

La théorie des jeux ordinaire décrit des agents *dans* des règles. La strate méta demande ce qui se passe quand **déplacer le jeu est une action payée** : quel trajet, quels murs franchis, à quel prix, et payé par qui. Sur l'univers fini de Robinson-Gofforth, chaque pièce de cette question est calculable -- ce notebook les calcule **ensemble**, sur des jeux portant un nom (Dilemme, Poule, Cerf), pour que le résultat se lise comme un récit et pas comme une coordonnée.

### 4.0. Le substrat étendu, hérité de GT-3b / GT-20

Un jeu = **deux tables de rangs**, une par joueur, chacune un 4-uplet ordonnant strictement les quatre cases (haut-gauche, haut-droite, bas-gauche, bas-droite) -- rang 4 = meilleur. Les **swaps** échangent les cases portant deux rangs adjacents `k` et `k+1`, de part et d'autre : `R1, R2, R3` (côté Ligne) et `C1, C2, C3` (côté Colonne). Ce sont les six générateurs de l'espace ; chaque traversée d'un swap franchit un **mur** du monde de Bruns-Kimmich -- c'est l'objet de la section 2. Toute la mécanique est reprise telle quelle des trois versants, sans modification : ce notebook est un consommateur du substrat, pas une refonte.

In [12]:
use std::collections::BTreeSet;

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));

// Substrat GT-3b / GT-3e / GT-20, repris tel quel -- pur stdlib

let mut set27: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set27.insert((a, b, c, d)); }
} } } }
let stricts: Vec<Tab> = set27.into_iter().collect();
let chambres27: Vec<Jeu> = stricts.iter().flat_map(|&r| stricts.iter().map(move |&c| (r, c))).collect();
println!("Tables strictes par joueur : {} | chambres (jeux stricts) : {}", stricts.len(), chambres27.len());
println!("Générateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)");
let poule: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let cerf: Jeu = ((4, 1, 3, 2), (4, 3, 1, 2));
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let renverse: Jeu = ((4, 3, 2, 1), (4, 3, 2, 1));
for (nom, j) in [("PD (Dilemme)", pd), ("POULE (Poule)", poule), ("CERF (Cerf)", cerf),
                 ("IDENTITE", identite), ("RENVERSE", renverse)] {
    let ok = chambres27.contains(&j);
    println!("  {:<16} Ligne {} | Colonne {} | chambre stricte : {}", nom, py_tup4(j.0), py_tup4(j.1), py_bool(ok));
};


Tables strictes par joueur : 24 | chambres (jeux stricts) : 576


Générateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)


  PD (Dilemme)     Ligne (3, 1, 4, 2) | Colonne (3, 4, 1, 2) | chambre stricte : True


  POULE (Poule)    Ligne (3, 2, 4, 1) | Colonne (3, 4, 2, 1) | chambre stricte : True


  CERF (Cerf)      Ligne (4, 1, 3, 2) | Colonne (4, 3, 1, 2) | chambre stricte : True


  IDENTITE         Ligne (1, 2, 3, 4) | Colonne (1, 2, 3, 4) | chambre stricte : True


  RENVERSE         Ligne (4, 3, 2, 1) | Colonne (4, 3, 2, 1) | chambre stricte : True


### Lecture du substrat

Vingt-quatre tables par joueur, donc 24 x 24 = **576 chambres** -- l'univers de Robinson-Gofforth classique, déjà dérivé par GT-20 et non récité ici. Les cinq bornes canoniques y vivent : le Dilemme du Prisonnier (défection mutuelle stable), la Poule (le jeu du poulet), la Chasse au Cerf (coordination sur le meilleur monde commun), et les deux extrêmes structurants Identité et Renversement.

Deux précisions de vocabulaire pour la suite. Un **pas** relie deux chambres adjacentes : il modifie exactement une table, celle du joueur qui agit, par exactement un swap de niveau `k`. Et une **méta-action** (GT-3e) est ce pas *considéré comme payé* : réécrire une préférence déclarée coûte quelque chose, en échelons de rang -- c'est la section 3 qui fixe le barème.

## 5. Le chemin que le lecteur n'écrit pas

Le premier impératif du parcours : le chemin ne doit pas être écrit à la main. Si le lecteur choisissait lui-même ses swaps, le notebook ne montrerait rien -- il redirait ses propres intuitions. Le constructeur ci-dessous produit donc le chemin par **BFS avec remontée des parents**, exactement selon le patron de GT-20 : le graphe des 576 chambres est exploré depuis le départ, et la chaîne est reconstruite en remontant de l'arrivée.

Le second impératif est la **séparation constructeur / vérificateur** (loi II de la série) : celui qui produit le témoin n'est pas celui qui le juge. La cellule suivante héberge un vérificateur qui ne réutilise *rien* du constructeur -- il re-dérive les distances par sa propre recherche exhaustive.

In [13]:
use std::collections::{BTreeMap, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    swap_val(t, k)
}

fn swap_jeu(jeu: Jeu, cote: usize, k: usize) -> Jeu {
    // cote 0 = "ligne", cote 1 = "colonne"
    if cote == 0 {
        (swap_valeurs_adjacentes(jeu.0, k), jeu.1)
    } else {
        (jeu.0, swap_valeurs_adjacentes(jeu.1, k))
    }
}

fn jeu_tab(j: Jeu, ti: usize) -> Tab {
    if ti == 0 { j.0 } else { j.1 }
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn cote_str(cote: usize) -> &'static str {
    if cote == 0 { "ligne" } else { "colonne" }
}

fn nommer_pas(pred: Jeu, cur: Jeu) -> String {
    for cote in 0..2usize {
        if jeu_tab(pred, cote) != jeu_tab(cur, cote) {
            for k in 1..4usize {
                if swap_jeu(pred, cote, k) == cur {
                    let nom_cote = if cote == 0 { "R" } else { "C" };
                    return format!("{}{} (côté {}, niveaux {} <-> {})", nom_cote, k, cote_str(cote), k, k + 1);
                }
            }
        }
    }
    "?".to_string()
}

fn construire_chemin(depart: Jeu, arrivee: Jeu) -> Option<Vec<(Jeu, Jeu)>> {
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    parent.insert(depart, None);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        if u == arrivee { break; }
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if !parent.contains_key(&v) {
                    parent.insert(v, Some(u));
                    q.push_back(v);
                }
            }
        }
    }
    if !parent.contains_key(&arrivee) { return None; }
    let mut chaine = Vec::new();
    let mut cur = arrivee;
    while let Some(Some(p)) = parent.get(&cur) {
        chaine.push((*p, cur));
        cur = *p;
    }
    chaine.reverse();
    Some(chaine)
}

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let poule: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let cerf: Jeu = ((4, 1, 3, 2), (4, 3, 1, 2));
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let renverse: Jeu = ((4, 3, 2, 1), (4, 3, 2, 1));

// === Section 1.1 : construire_chemin -- le constructeur de témoin (patron GT-20) ===

let chemin_pd_cerf = construire_chemin(pd, cerf).unwrap();
println!("Chemin construit : Dilemme du Prisonnier -> Chasse au Cerf");
println!("  départ  PD   : Ligne {} | Colonne {}", py_tup4(pd.0), py_tup4(pd.1));
for &(pred, cur) in &chemin_pd_cerf {
    println!("  -- {} -->  Ligne {} | Colonne {}", nommer_pas(pred, cur), py_tup4(cur.0), py_tup4(cur.1));
}
println!("Longueur du chemin produit : {} pas", chemin_pd_cerf.len());


Chemin construit : Dilemme du Prisonnier -> Chasse au Cerf


  départ  PD   : Ligne (3, 1, 4, 2) | Colonne (3, 4, 1, 2)


  -- R3 (côté ligne, niveaux 3 <-> 4) -->  Ligne (4, 1, 3, 2) | Colonne (3, 4, 1, 2)


  -- C3 (côté colonne, niveaux 3 <-> 4) -->  Ligne (4, 1, 3, 2) | Colonne (4, 3, 1, 2)


Longueur du chemin produit : 2 pas


### Lecture : deux révisions du sommet

Le chemin construit a **2 pas** -- et chacun réécrit le **sommet** des préférences d'un joueur (niveaux 3 <-> 4). Ce n'est pas un choix du constructeur, c'est la géométrie : pour passer du Dilemme au Cerf, Ligne doit échanger ses deux meilleures cases (sa table `(3, 1, 4, 2)` devient `(4, 1, 3, 2)`) et Colonne doit faire de même. En langage des noms : dans le Dilemme, la défection est le sommet de chacun ; dans le Cerf, c'est la coordination. Aucun détour par les bas niveaux ne raccourcit ce trajet -- il faut toucher au sommet, deux fois.

On notera l'asymétrie apparente des tables du Cerf (`(4, 1, 3, 2)` et `(4, 3, 1, 2)`) : les deux joueurs y placent le même sommet mais ne classent pas pareil les deux pires cases -- le Cerf n'exige pas d'êtres identiques, seulement de préférer le monde commun.

In [14]:
use std::collections::{BTreeMap, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    swap_val(t, k)
}

fn swap_jeu(jeu: Jeu, cote: usize, k: usize) -> Jeu {
    // cote 0 = "ligne", cote 1 = "colonne"
    if cote == 0 {
        (swap_valeurs_adjacentes(jeu.0, k), jeu.1)
    } else {
        (jeu.0, swap_valeurs_adjacentes(jeu.1, k))
    }
}

fn jeu_tab(j: Jeu, ti: usize) -> Tab {
    if ti == 0 { j.0 } else { j.1 }
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn cote_str(cote: usize) -> &'static str {
    if cote == 0 { "ligne" } else { "colonne" }
}

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn bfs_complet(depart: Jeu) -> BTreeMap<Jeu, usize> {
    let mut dist = BTreeMap::new();
    dist.insert(depart, 0);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if !dist.contains_key(&v) {
                    dist.insert(v, du + 1);
                    q.push_back(v);
                }
            }
        }
    }
    dist
}

fn pas_elementaire_valide(pred: Jeu, cur: Jeu) -> bool {
    (0..2usize).any(|cote| (1..4usize).any(|k| swap_jeu(pred, cote, k) == cur))
}

fn verifier_chemin(depart: Jeu, arrivee: Jeu, chaine: Option<&[(Jeu, Jeu)]>) -> String {
    let ch = match chaine {
        None => return "INVALIDE : chemin vide".to_string(),
        Some(c) => c,
    };
    if ch.is_empty() {
        return "INVALIDE : chemin vide".to_string();
    }
    if ch[0].0 != depart || ch[ch.len() - 1].1 != arrivee {
        return "INVALIDE : les extrémités ne sont pas celles annoncées".to_string();
    }
    for &(pred, cur) in ch {
        if !pas_elementaire_valide(pred, cur) {
            return format!("INVALIDE : pas non élémentaire vers {}", py_jeu(cur));
        }
    }
    let d_reel = bfs_complet(depart)[&arrivee];
    if ch.len() != d_reel {
        return format!("VALIDE mais NON MINIMAL : {} pas, distance réelle {}", ch.len(), d_reel);
    }
    format!("VALIDE + MINIMAL : {} pas = distance exhaustive d(G,H)", ch.len())
}

fn construire_chemin(depart: Jeu, arrivee: Jeu) -> Option<Vec<(Jeu, Jeu)>> {
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    parent.insert(depart, None);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        if u == arrivee { break; }
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if !parent.contains_key(&v) {
                    parent.insert(v, Some(u));
                    q.push_back(v);
                }
            }
        }
    }
    if !parent.contains_key(&arrivee) { return None; }
    let mut chaine = Vec::new();
    let mut cur = arrivee;
    while let Some(Some(p)) = parent.get(&cur) {
        chaine.push((*p, cur));
        cur = *p;
    }
    chaine.reverse();
    Some(chaine)
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn py_jeu(j: Jeu) -> String {
    format!("({}, {})", py_tup4(j.0), py_tup4(j.1))
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let poule: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let cerf: Jeu = ((4, 1, 3, 2), (4, 3, 1, 2));
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let renverse: Jeu = ((4, 3, 2, 1), (4, 3, 2, 1));

let chemin_pd_cerf = construire_chemin(pd, cerf).unwrap();

// === Section 1.2 : verifier_chemin -- le vérificateur indépendant (patron GT-20) ===

println!("1. Le chemin du constructeur (PD -> CERF) :");
println!("   {}", verifier_chemin(pd, cerf, Some(&chemin_pd_cerf)));
let voisin_cerf = swap_jeu(cerf, 0, 1);
let mut chemin_detour = chemin_pd_cerf.clone();
chemin_detour.push((cerf, voisin_cerf));
chemin_detour.push((voisin_cerf, cerf));
println!("2. Le même chemin rallongé d'un aller-retour :");
println!("   {}", verifier_chemin(pd, cerf, Some(&chemin_detour)));
println!("3. Un 'chemin' en un pas vers un jeu non adjacent :");
println!("   {}", verifier_chemin(pd, poule, Some(&[(pd, poule)])));


1. Le chemin du constructeur (PD -> CERF) :


   VALIDE + MINIMAL : 2 pas = distance exhaustive d(G,H)


2. Le même chemin rallongé d'un aller-retour :


   VALIDE mais NON MINIMAL : 4 pas, distance réelle 2


3. Un 'chemin' en un pas vers un jeu non adjacent :


   INVALIDE : pas non élémentaire vers ((3, 2, 4, 1), (3, 4, 2, 1))


### Lecture : trois verdicts, une séparation

Le vérificateur rend trois verdicts distincts sur trois cas construits pour eux : `VALIDE + MINIMAL` pour le témoin du constructeur, `NON MINIMAL` pour le même chemin artificiellement rallongé d'un aller-retour, et `INVALIDE` pour un saut direct PD -> POULE (deux jeux que rien ne relie en un pas). La minimalité n'est pas une impression : elle est **re-dérivée par recherche exhaustive** depuis le départ, indépendamment de la façon dont le constructeur a trouvé son chemin.

C'est la loi II de la série, appliquée au parcours : le constructeur (BFS + parents) *produit* un témoin, le vérificateur (re-dérivation complète) le *juge*. À la section 4, ce couple sera étendu aux murs et aux coûts : le vérificateur final re-vérifiera tout ce que le parcours affiche.

## 6. Voir les murs

Entre deux chambres adjacentes -- avant et après un swap de niveau `k` -- il y a un **mur** : un jeu à rangs où les deux niveaux `k` et `k+1` sont tombés **ex æquo**. C'est la lecture Bruns-Kimmich reprise de GT-3b : le monde complet des jeux à rangs compte 5625 points, dont 576 chambres (aucune égalité) et des murs de codimension 1 (exactement une paire ex æquo). Traverser un swap, c'est littéralement *passer à travers* le mur qui sépare les deux chambres.

Deux opérations réciproques le formalisent : **fusionner** les niveaux `k` et `k+1` d'une chambre donne le mur qu'un pas de niveau `k` traverse ; **briser le tie** d'un mur redonne les deux chambres qu'il sépare. La propriété qui fait tenir la section : briser le tie du mur d'un pas doit redonner *exactement* les deux tables extrémités de ce pas. Elle est vérifiée globalement ci-dessous, sur les 24 tables et les 3 niveaux.

In [15]:
use std::collections::BTreeSet;

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    swap_val(t, k)
}

fn mur_du_niveau(t: Tab, k: usize) -> Tab {
    let mut l = [0usize; 4];
    for i in 0..4 {
        l[i] = if tab_get(t, i) > k { tab_get(t, i) - 1 } else { tab_get(t, i) };
    }
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    l[pk] = k;
    l[pk1] = k;
    (l[0], l[1], l[2], l[3])
}

fn briser_le_tie(t: Tab) -> (Tab, Tab) {
    // la valeur ex aequo (mur simple : unique)
    let mut v = usize::MAX;
    for i in 0..4 {
        let x = tab_get(t, i);
        if (0..4).any(|j| j != i && tab_get(t, j) == x) { v = v.min(x); }
    }
    let ij: Vec<usize> = (0..4).filter(|&i| tab_get(t, i) == v).collect();
    let (i, j) = (ij[0], ij[1]);
    let mut inc = [0usize; 4];
    for q in 0..4 {
        inc[q] = if tab_get(t, q) > v { tab_get(t, q) + 1 } else { tab_get(t, q) };
    }
    let mut a = inc;
    a[i] = v; a[j] = v + 1;
    let mut b = inc;
    b[i] = v + 1; b[j] = v;
    ((a[0], a[1], a[2], a[3]), (b[0], b[1], b[2], b[3]))
}

fn nb_uniques(t: Tab) -> usize {
    BTreeSet::from([t.0, t.1, t.2, t.3]).len()
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

let mut set27: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set27.insert((a, b, c, d)); }
} } } }
let stricts: Vec<Tab> = set27.into_iter().collect();
let chambres27: Vec<Jeu> = stricts.iter().flat_map(|&r| stricts.iter().map(move |&c| (r, c))).collect();

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let poule: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let cerf: Jeu = ((4, 1, 3, 2), (4, 3, 1, 2));
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let renverse: Jeu = ((4, 3, 2, 1), (4, 3, 2, 1));

// === Section 2.1 : le mur d'un pas -- fusion de niveaux et brisure de tie (GT-3b) ===

let prop = stricts.iter().all(|&t| (1..4usize).all(|k| {
    let f = briser_le_tie(mur_du_niveau(t, k));
    let mut faces = vec![f.0, f.1];
    faces.sort();
    let mut attendu = vec![t, swap_valeurs_adjacentes(t, k)];
    attendu.sort();
    faces == attendu
}));
println!("Propriété mur <-> paire de chambres, vérifiée sur 24 tables x 3 niveaux : {}", py_bool(prop));
let mut murs_par_joueur: BTreeSet<Tab> = BTreeSet::new();
for &t in &stricts { for k in 1..4usize { murs_par_joueur.insert(mur_du_niveau(t, k)); } }
println!("Murs simples distincts par joueur : {} (GT-3b mesurait 36)", murs_par_joueur.len());
let (t35, k35) = (pd.0, 3usize);
let mur35 = mur_du_niveau(t35, k35);
let faces35 = briser_le_tie(mur35);
println!("Exemple : table Ligne du Dilemme {} | pas de niveau {}", py_tup4(t35), k35);
println!("  mur          : {} (codim {})", py_tup4(mur35), 4 - nb_uniques(mur35));
println!("  faces du mur : {}, {}", py_tup4(faces35.0), py_tup4(faces35.1));
println!("  extrémités   : {} et {}", py_tup4(t35), py_tup4(swap_valeurs_adjacentes(t35, 3)));


Propriété mur <-> paire de chambres, vérifiée sur 24 tables x 3 niveaux : True


Murs simples distincts par joueur : 36 (GT-3b mesurait 36)


Exemple : table Ligne du Dilemme (3, 1, 4, 2) | pas de niveau 3


  mur          : (3, 1, 3, 2) (codim 1)


  faces du mur : (3, 1, 4, 2), (4, 1, 3, 2)


  extrémités   : (3, 1, 4, 2) et (4, 1, 3, 2)


### Lecture : le mur a deux faces, et ce sont les extrémités du pas

La propriété est vérifiée sur les 72 combinaisons (24 tables x 3 niveaux) : **briser le tie du mur redonne exactement la chambre de départ et la chambre d'arrivée du pas**. Le mur n'est donc pas une métaphore décorative -- c'est l'objet géométrique *entre* les deux jeux, celui dont les deux faces sont les extrémités du pas. Sur l'exemple affiché : la table Ligne du Dilemme `(3, 1, 4, 2)` et sa voisine de niveau 3 `(4, 1, 3, 2)` -- la table du Cerf -- sont les deux faces du mur `(3, 1, 3, 2)`, où les cases haut-gauche et bas-gauche sont ex æquo au sommet.

Le comptage rejoint GT-3b : **36 murs simples par joueur** (24 chambres x 3 niveaux, chaque mur compté deux fois car chaque mur touche exactement 2 chambres). Chaque niveau de préférence a ses murs ; franchir un mur de niveau 3, c'est réécrire un sommet -- le lien avec le coût de la section suivante est direct.

In [16]:
use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    swap_val(t, k)
}

fn swap_jeu(jeu: Jeu, cote: usize, k: usize) -> Jeu {
    // cote 0 = "ligne", cote 1 = "colonne"
    if cote == 0 {
        (swap_valeurs_adjacentes(jeu.0, k), jeu.1)
    } else {
        (jeu.0, swap_valeurs_adjacentes(jeu.1, k))
    }
}

fn jeu_tab(j: Jeu, ti: usize) -> Tab {
    if ti == 0 { j.0 } else { j.1 }
}

fn cote_str(cote: usize) -> &'static str {
    if cote == 0 { "ligne" } else { "colonne" }
}

fn nommer_pas(pred: Jeu, cur: Jeu) -> String {
    for cote in 0..2usize {
        if jeu_tab(pred, cote) != jeu_tab(cur, cote) {
            for k in 1..4usize {
                if swap_jeu(pred, cote, k) == cur {
                    let nom_cote = if cote == 0 { "R" } else { "C" };
                    return format!("{}{} (côté {}, niveaux {} <-> {})", nom_cote, k, cote_str(cote), k, k + 1);
                }
            }
        }
    }
    "?".to_string()
}

fn nb_uniques(t: Tab) -> usize {
    BTreeSet::from([t.0, t.1, t.2, t.3]).len()
}

fn mur_du_niveau(t: Tab, k: usize) -> Tab {
    let mut l = [0usize; 4];
    for i in 0..4 {
        l[i] = if tab_get(t, i) > k { tab_get(t, i) - 1 } else { tab_get(t, i) };
    }
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    l[pk] = k;
    l[pk1] = k;
    (l[0], l[1], l[2], l[3])
}

fn briser_le_tie(t: Tab) -> (Tab, Tab) {
    // la valeur ex aequo (mur simple : unique)
    let mut v = usize::MAX;
    for i in 0..4 {
        let x = tab_get(t, i);
        if (0..4).any(|j| j != i && tab_get(t, j) == x) { v = v.min(x); }
    }
    let ij: Vec<usize> = (0..4).filter(|&i| tab_get(t, i) == v).collect();
    let (i, j) = (ij[0], ij[1]);
    let mut inc = [0usize; 4];
    for q in 0..4 {
        inc[q] = if tab_get(t, q) > v { tab_get(t, q) + 1 } else { tab_get(t, q) };
    }
    let mut a = inc;
    a[i] = v; a[j] = v + 1;
    let mut b = inc;
    b[i] = v + 1; b[j] = v;
    ((a[0], a[1], a[2], a[3]), (b[0], b[1], b[2], b[3]))
}

fn construire_chemin(depart: Jeu, arrivee: Jeu) -> Option<Vec<(Jeu, Jeu)>> {
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    parent.insert(depart, None);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        if u == arrivee { break; }
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if !parent.contains_key(&v) {
                    parent.insert(v, Some(u));
                    q.push_back(v);
                }
            }
        }
    }
    if !parent.contains_key(&arrivee) { return None; }
    let mut chaine = Vec::new();
    let mut cur = arrivee;
    while let Some(Some(p)) = parent.get(&cur) {
        chaine.push((*p, cur));
        cur = *p;
    }
    chaine.reverse();
    Some(chaine)
}

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let poule: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let cerf: Jeu = ((4, 1, 3, 2), (4, 3, 1, 2));
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let renverse: Jeu = ((4, 3, 2, 1), (4, 3, 2, 1));

let chemin_pd_cerf = construire_chemin(pd, cerf).unwrap();

// === Section 2.2 : les murs du chemin PD -> CERF, pas à pas ===

println!("Le chemin PD -> CERF, lu mur par mur :");
for (n37, &(pred, cur)) in chemin_pd_cerf.iter().enumerate() {
    let (cote, ti) = if pred.0 != cur.0 { (0usize, 0usize) } else { (1, 1) };
    let k = (1..4usize).find(|&kk| swap_jeu(pred, cote, kk) == cur).unwrap();
    let mur = mur_du_niveau(jeu_tab(pred, ti), k);
    let faces = briser_le_tie(mur);
    let mut f2 = vec![faces.0, faces.1];
    f2.sort();
    let mut att = vec![jeu_tab(pred, ti), jeu_tab(cur, ti)];
    att.sort();
    let confirmee = f2 == att;
    println!("  pas {} : {}", n37 + 1, nommer_pas(pred, cur));
    println!("     table {} : {} -> {}", cote_str(cote), py_tup4(jeu_tab(pred, ti)), py_tup4(jeu_tab(cur, ti)));
    println!("     mur traversé : {} (codim {}) | faces : {}, {}", py_tup4(mur), 4 - nb_uniques(mur), py_tup4(faces.0), py_tup4(faces.1));
    println!("     les faces sont les extrémités du pas : {}", py_bool(confirmee));
};


Le chemin PD -> CERF, lu mur par mur :


  pas 1 : R3 (côté ligne, niveaux 3 <-> 4)


     table ligne : (3, 1, 4, 2) -> (4, 1, 3, 2)


     mur traversé : (3, 1, 3, 2) (codim 1) | faces : (3, 1, 4, 2), (4, 1, 3, 2)


     les faces sont les extrémités du pas : True


  pas 2 : C3 (côté colonne, niveaux 3 <-> 4)


     table colonne : (3, 4, 1, 2) -> (4, 3, 1, 2)


     mur traversé : (3, 3, 1, 2) (codim 1) | faces : (3, 4, 1, 2), (4, 3, 1, 2)


     les faces sont les extrémités du pas : True


### Lecture : deux murs, un par joueur

Le chemin PD -> CERF traverse exactement **deux murs**, un par joueur, tous deux de niveau 3. Le mur de Ligne, `(3, 1, 3, 2)`, est le jeu où Ligne est *indifférent entre défection et coopération* (ses cases haut-gauche et bas-gauche ex æquo au sommet) tandis que la table de Colonne est encore celle du Dilemme. Le mur de Colonne, `(3, 3, 1, 2)`, est le miroir : Colonne indifférent au sommet, Ligne déjà converti au Cerf.

C'est le contenu concret du critère d'intégration : le chemin n'est plus une suite de coordonnées, c'est une suite de **situations intermédiaires lisibles** -- chacune est un jeu où un joueur exactement est suspendu entre ses deux anciens idéaux. La ligne `confirmee : True` de chaque pas est la garantie que le mur exhibé est bien *celui-là* et pas un voisin.

## 7. Le coût des méta-actions

Chaque pas du chemin est une **méta-action** au sens de GT-3e : un joueur réécrit ses préférences déclarées, et cette réécriture est payée en échelons de rang -- l'unité honnête d'un monde purement ordinal (payer 1, c'est renoncer à un échelon pour atteindre le niveau visé). Reste à fixer le **barème**. Deux lectures naturelles :

- **par côté** : Ligne paie `c_L` par swap, Colonne `c_C` -- le tarif différencié, par exemple la réécriture de la table Colonne (l'"institution") plus cher que celle de Ligne ;
- **par niveau** : réécrire le bas de sa liste coûte 1, le milieu 2, le **sommet 3** -- réviser ce qu'on préfère *entre tout* est la révision la plus profonde.

La question qui semble s'imposer : le chemin le plus court en nombre de pas est-il le moins cher ? L'intuition crie *non* -- un détour par des niveaux bon marché pourrait battre le trajet direct. La mesure, elle, répond autrement.

In [17]:
use std::collections::{BTreeMap, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn dist_tables(t0: Tab) -> BTreeMap<Tab, usize> {
    bfs_from(t0)
}

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let poule: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let cerf: Jeu = ((4, 1, 3, 2), (4, 3, 1, 2));
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let renverse: Jeu = ((4, 3, 2, 1), (4, 3, 2, 1));

// === Section 3.1 : tarifs par côté -- le coût est géométrique ===

let noms40 = [("PD", pd), ("POULE", poule), ("CERF", cerf), ("IDENTITE", identite), ("RENVERSE", renverse)];
let (c_l, c_c) = (1i64, 3i64);
println!("Tarifs par côté : Ligne {}, Colonne {}", c_l, c_c);
println!("{:>22} | d_L | d_C | longueur | coût | Ligne paie | Colonne paie", "paire");
println!("{}", "-".repeat(78));
for i in 0..noms40.len() {
    for j in i + 1..noms40.len() {
        let (n1, j1) = noms40[i];
        let (n2, j2) = noms40[j];
        let dl = dist_tables(j1.0)[&j2.0];
        let dc = dist_tables(j1.1)[&j2.1];
        println!("{:>10} -> {:<10} | {}  | {}  |    {}     |  {:>2}  |     {}      |      {}",
            n1, n2, dl, dc, dl + dc, c_l * dl as i64 + c_c * dc as i64, c_l * dl as i64, c_c * dc as i64);
    }
};


Tarifs par côté : Ligne 1, Colonne 3


                 paire | d_L | d_C | longueur | coût | Ligne paie | Colonne paie


------------------------------------------------------------------------------


        PD -> POULE      | 1  | 1  |    2     |   4  |     1      |      3


        PD -> CERF       | 1  | 1  |    2     |   4  |     1      |      3


        PD -> IDENTITE   | 3  | 4  |    7     |  15  |     3      |      12


... (3 outputs left out)

     POULE -> RENVERSE   | 2  | 1  |    3     |   5  |     2      |      3


      CERF -> IDENTITE   | 4  | 5  |    9     |  19  |     4      |      15


      CERF -> RENVERSE   | 2  | 1  |    3     |   5  |     2      |      3


  IDENTITE -> RENVERSE   | 6  | 6  |    12     |  24  |     6      |      18


### Lecture : un théorème, pas une coïncidence

Les colonnes `d_L` et `d_C` sont les distances **dans le graphe des 24 tables d'un seul joueur** -- chacune recalculée par BFS indépendant. Et le tableau montre la structure : **la longueur du chemin minimal est exactement `d_L + d_C`**, et son coût exactement `c_L·d_L + c_C·d_C`. Ce n'est pas un accident d'échantillon, c'est un théorème de trois lignes :

> chaque pas modifie exactement **une** table ; la table de Ligne doit passer de `PD[0]` à `CERF[0]`, ce qui exige au moins `d_L` pas côté Ligne ; symétriquement au moins `d_C` côté Colonne. Tout chemin a donc au moins `d_L + d_C` pas et coûte au moins `c_L·d_L + c_C·d_C` -- et le chemin BFS atteint exactement ces deux bornes.

La conséquence est le résultat central de cette section : **le coût du trajet minimal n'est pas négociable**. Aucun choix de chemin -- aucune ruse, aucun détour -- ne change la facture : qui paie quoi est fixé par la géométrie des deux tables, avant tout parcours. Pour aller du Dilemme au Cerf à ces tarifs, Ligne paie `c_L` et Colonne `c_C` -- quel que soit le chemin minimal emprunté.

In [18]:
use std::cmp::Reverse;
use std::collections::{BTreeMap, BTreeSet, BinaryHeap, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    swap_val(t, k)
}

fn swap_jeu(jeu: Jeu, cote: usize, k: usize) -> Jeu {
    // cote 0 = "ligne", cote 1 = "colonne"
    if cote == 0 {
        (swap_valeurs_adjacentes(jeu.0, k), jeu.1)
    } else {
        (jeu.0, swap_valeurs_adjacentes(jeu.1, k))
    }
}

fn jeu_tab(j: Jeu, ti: usize) -> Tab {
    if ti == 0 { j.0 } else { j.1 }
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn cote_str(cote: usize) -> &'static str {
    if cote == 0 { "ligne" } else { "colonne" }
}

fn nommer_pas(pred: Jeu, cur: Jeu) -> String {
    for cote in 0..2usize {
        if jeu_tab(pred, cote) != jeu_tab(cur, cote) {
            for k in 1..4usize {
                if swap_jeu(pred, cote, k) == cur {
                    let nom_cote = if cote == 0 { "R" } else { "C" };
                    return format!("{}{} (côté {}, niveaux {} <-> {})", nom_cote, k, cote_str(cote), k, k + 1);
                }
            }
        }
    }
    "?".to_string()
}

fn construire_chemin(depart: Jeu, arrivee: Jeu) -> Option<Vec<(Jeu, Jeu)>> {
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    parent.insert(depart, None);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        if u == arrivee { break; }
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if !parent.contains_key(&v) {
                    parent.insert(v, Some(u));
                    q.push_back(v);
                }
            }
        }
    }
    if !parent.contains_key(&arrivee) { return None; }
    let mut chaine = Vec::new();
    let mut cur = arrivee;
    while let Some(Some(p)) = parent.get(&cur) {
        chaine.push((*p, cur));
        cur = *p;
    }
    chaine.reverse();
    Some(chaine)
}

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn cout_chemin(chaine: &[(Jeu, Jeu)], w: &BTreeMap<usize, i64>) -> i64 {
    let mut total = 0i64;
    for &(pred, cur) in chaine {
        let (cote, _) = if pred.0 != cur.0 { (0usize, 0usize) } else { (1, 1) };
        let k = (1..4usize).find(|&kk| swap_jeu(pred, cote, kk) == cur).unwrap();
        total += w[&k];
    }
    total
}

fn chemin_min_cout(depart: Jeu, arrivee: Jeu, w: &BTreeMap<usize, i64>) -> Vec<(Jeu, Jeu)> {
    let inf = 1_000_000_000i64;
    let mut dist: BTreeMap<Jeu, i64> = BTreeMap::new();
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    let mut pq: BinaryHeap<Reverse<(i64, Jeu)>> = BinaryHeap::new();
    dist.insert(depart, 0);
    parent.insert(depart, None);
    pq.push(Reverse((0, depart)));
    while let Some(Reverse((d, u))) = pq.pop() {
        if u == arrivee { break; }
        if d > *dist.get(&u).unwrap_or(&inf) { continue; }
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if d + w[&k] < *dist.get(&v).unwrap_or(&inf) {
                    dist.insert(v, d + w[&k]);
                    parent.insert(v, Some(u));
                    pq.push(Reverse((d + w[&k], v)));
                }
            }
        }
    }
    let mut chaine = Vec::new();
    let mut cur = arrivee;
    while let Some(Some(p)) = parent.get(&cur) {
        chaine.push((*p, cur));
        cur = *p;
    }
    chaine.reverse();
    chaine
}

fn py_list_str(v: &[String]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}

// MT19937 compatible CPython : random.seed(0) niveau module = random.Random(0)
// = init_by_array([0]). Arithmetique u32 a debordement. A recopier tel quel (cellule 42).
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {                         // i continue la ou la boucle precedente l'a laisse
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 32 ici : mots tronques en tete
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // n >= 1 ; rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
}

let mut set27: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize { for b in 1..=4usize { for c in 1..=4usize { for d in 1..=4usize {
    let s = BTreeSet::from([a, b, c, d]);
    if s.len() == 4 { set27.insert((a, b, c, d)); }
} } } }
let stricts: Vec<Tab> = set27.into_iter().collect();
let chambres27: Vec<Jeu> = stricts.iter().flat_map(|&r| stricts.iter().map(move |&c| (r, c))).collect();

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let poule: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let cerf: Jeu = ((4, 1, 3, 2), (4, 3, 1, 2));
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let renverse: Jeu = ((4, 3, 2, 1), (4, 3, 2, 1));

let noms40 = [("PD", pd), ("POULE", poule), ("CERF", cerf), ("IDENTITE", identite), ("RENVERSE", renverse)];

// === Section 3.2 : tarifs par niveau -- la mesure répond à l'intuition ===

let w: BTreeMap<usize, i64> = BTreeMap::from([(1, 1), (2, 2), (3, 3)]);
println!("Tarifs par niveau : {:?} | le plus court chemin est-il le moins cher ?", w);
let mut ecarts = 0;
for i in 0..noms40.len() {
    for j in i + 1..noms40.len() {
        let (n1, j1) = noms40[i];
        let (n2, j2) = noms40[j];
        let bfs = construire_chemin(j1, j2).unwrap();
        let dij = chemin_min_cout(j1, j2, &w);
        let cb = cout_chemin(&bfs, &w);
        let cd = cout_chemin(&dij, &w);
        let ecart = cb - cd;
        if ecart > 0 { ecarts += 1; }
        println!("  {:>8} -> {:<8} : BFS {} pas à {} | Dijkstra {} pas à {} | écart {}", n1, n2, bfs.len(), cb, dij.len(), cd, ecart);
    }
}
println!("Paires nommées où un détour battrait le plus court chemin : {} / 10", ecarts);
let mut g = Mt19937::new(0);
let mut ech: Vec<(Jeu, Jeu)> = Vec::new();
for _ in 0..300 {
    let a = *g.choice(&chambres27);
    let b = *g.choice(&chambres27);
    ech.push((a, b));
}
let mut disc = 0;
for &(a, b) in &ech {
    let dij = chemin_min_cout(a, b, &w);
    let bfs = construire_chemin(a, b).unwrap();
    if cout_chemin(&dij, &w) < cout_chemin(&bfs, &w) { disc += 1; }
}
println!("Échantillon de 300 paires quelconques de chambres : {} détour(s) gagnant(s)", disc);
let bfs_pi = construire_chemin(pd, identite).unwrap();
let dij_pi = chemin_min_cout(pd, identite, &w);
println!();
println!("PD -> IDENTITE : le BFS et le Dijkstra produisent deux chemins distincts");
let noms_bfs: Vec<String> = bfs_pi.iter().take(4).map(|&(p, c)| nommer_pas(p, c)).collect();
println!("  BFS       : {} ... coût {}", py_list_str(&noms_bfs), cout_chemin(&bfs_pi, &w));
let noms_dij: Vec<String> = dij_pi.iter().take(4).map(|&(p, c)| nommer_pas(p, c)).collect();
println!("  Dijkstra  : {} ... coût {}", py_list_str(&noms_dij), cout_chemin(&dij_pi, &w));


Tarifs par niveau : {1: 1, 2: 2, 3: 3} | le plus court chemin est-il le moins cher ?


        PD -> POULE    : BFS 2 pas à 2 | Dijkstra 2 pas à 2 | écart 0


        PD -> CERF     : BFS 2 pas à 6 | Dijkstra 2 pas à 6 | écart 0


        PD -> IDENTITE : BFS 7 pas à 14 | Dijkstra 7 pas à 14 | écart 0


        PD -> RENVERSE : BFS 5 pas à 10 | Dijkstra 5 pas à 10 | écart 0


     POULE -> CERF     : BFS 4 pas à 8 | Dijkstra 4 pas à 8 | écart 0


... (6 outputs left out)

Échantillon de 300 paires quelconques de chambres : 0 détour(s) gagnant(s)


PD -> IDENTITE : le BFS et le Dijkstra produisent deux chemins distincts


  BFS       : ['R2 (côté ligne, niveaux 2 <-> 3)', 'R1 (côté ligne, niveaux 1 <-> 2)', 'R3 (côté ligne, niveaux 3 <-> 4)', 'C2 (côté colonne, niveaux 2 <-> 3)'] ... coût 14


  Dijkstra  : ['R2 (côté ligne, niveaux 2 <-> 3)', 'R1 (côté ligne, niveaux 1 <-> 2)', 'C2 (côté colonne, niveaux 2 <-> 3)', 'C1 (côté colonne, niveaux 1 <-> 2)'] ... coût 14


### Lecture : le plus court est aussi le moins cher -- mesuré, pas prouvé

L'intuition du détour économique est **réfutée par la mesure** : sur les 10 paires nommées et sur un échantillon de 300 paires quelconques de chambres, **aucun** chemin de coût minimal n'est plus long que le plus court chemin. Sous ces tarifs, la longueur minimale et le coût minimal sont atteints *simultanément* -- le Dijkstra et le BFS rendent la même facture. Ce fait est **mesuré**, pas démontré : la section 3.1 le prouve pour les tarifs par côté (théorème des bornes `d_L`, `d_C`), mais pour les tarifs par niveau, aucun argument court ne clôt la question -- c'est dit tel quel, et l'exercice 2 la rouvre.

Le dernier affichage montre l'autre face de la structure : pour PD -> IDENTITE, BFS et Dijkstra produisent **deux chemins distincts de même coût**. Le témoin n'est pas unique -- plusieurs routes mènent au même prix -- mais la facture, elle, ne varie pas. Réuni au théorème de la section 3.1, le paysage économique du parcours tient en une phrase : **le prix du voyage est un invariant géométrique ; le chemin, lui, est un choix parmi des égalités**.

## 8. Le parcours complet, bout en bout

Les trois pièces sont prêtes : le constructeur de chemin (section 1), les murs (section 2), le barème (section 3). La fonction ci-dessous les assemble dans **un seul bloc** -- celui que le critère d'intégration du chantier demande : un jeu nommé, un chemin construit et non écrit à la main, chaque mur traversé exhibé avec ses deux faces, chaque méta-action affichée avec son coût et son payeur, et la facture finale par joueur.

Le barème est celui de la section 3.2 (bas 1, milieu 2, sommet 3), le même pour les deux joueurs -- la profondeur de la révision fait le prix, pas l'identité de qui révise.

In [19]:
use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    swap_val(t, k)
}

fn swap_jeu(jeu: Jeu, cote: usize, k: usize) -> Jeu {
    // cote 0 = "ligne", cote 1 = "colonne"
    if cote == 0 {
        (swap_valeurs_adjacentes(jeu.0, k), jeu.1)
    } else {
        (jeu.0, swap_valeurs_adjacentes(jeu.1, k))
    }
}

fn jeu_tab(j: Jeu, ti: usize) -> Tab {
    if ti == 0 { j.0 } else { j.1 }
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn cote_str(cote: usize) -> &'static str {
    if cote == 0 { "ligne" } else { "colonne" }
}

fn nommer_pas(pred: Jeu, cur: Jeu) -> String {
    for cote in 0..2usize {
        if jeu_tab(pred, cote) != jeu_tab(cur, cote) {
            for k in 1..4usize {
                if swap_jeu(pred, cote, k) == cur {
                    let nom_cote = if cote == 0 { "R" } else { "C" };
                    return format!("{}{} (côté {}, niveaux {} <-> {})", nom_cote, k, cote_str(cote), k, k + 1);
                }
            }
        }
    }
    "?".to_string()
}

fn construire_chemin(depart: Jeu, arrivee: Jeu) -> Option<Vec<(Jeu, Jeu)>> {
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    parent.insert(depart, None);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        if u == arrivee { break; }
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if !parent.contains_key(&v) {
                    parent.insert(v, Some(u));
                    q.push_back(v);
                }
            }
        }
    }
    if !parent.contains_key(&arrivee) { return None; }
    let mut chaine = Vec::new();
    let mut cur = arrivee;
    while let Some(Some(p)) = parent.get(&cur) {
        chaine.push((*p, cur));
        cur = *p;
    }
    chaine.reverse();
    Some(chaine)
}

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn mur_du_niveau(t: Tab, k: usize) -> Tab {
    let mut l = [0usize; 4];
    for i in 0..4 {
        l[i] = if tab_get(t, i) > k { tab_get(t, i) - 1 } else { tab_get(t, i) };
    }
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    l[pk] = k;
    l[pk1] = k;
    (l[0], l[1], l[2], l[3])
}

fn briser_le_tie(t: Tab) -> (Tab, Tab) {
    // la valeur ex aequo (mur simple : unique)
    let mut v = usize::MAX;
    for i in 0..4 {
        let x = tab_get(t, i);
        if (0..4).any(|j| j != i && tab_get(t, j) == x) { v = v.min(x); }
    }
    let ij: Vec<usize> = (0..4).filter(|&i| tab_get(t, i) == v).collect();
    let (i, j) = (ij[0], ij[1]);
    let mut inc = [0usize; 4];
    for q in 0..4 {
        inc[q] = if tab_get(t, q) > v { tab_get(t, q) + 1 } else { tab_get(t, q) };
    }
    let mut a = inc;
    a[i] = v; a[j] = v + 1;
    let mut b = inc;
    b[i] = v + 1; b[j] = v;
    ((a[0], a[1], a[2], a[3]), (b[0], b[1], b[2], b[3]))
}

fn nb_uniques(t: Tab) -> usize {
    BTreeSet::from([t.0, t.1, t.2, t.3]).len()
}

fn parcours_complet(depart: Jeu, arrivee: Jeu, w: &BTreeMap<usize, i64>, nom_dep: &str, nom_arr: &str) -> Vec<(Jeu, Jeu)> {
    let chaine = construire_chemin(depart, arrivee).unwrap();
    println!("PARCOURS COMPLET : {} -> {}", nom_dep, nom_arr);
    println!("  barème par niveau : {:?} (le sommet coûte le plus cher)", w);
    println!("  {} : Ligne {} | Colonne {}", nom_dep, py_tup4(depart.0), py_tup4(depart.1));
    let mut cumul = 0i64;
    let (mut pl, mut pc) = (0i64, 0i64);
    for (n, &(pred, cur)) in chaine.iter().enumerate() {
        let (cote, ti) = if pred.0 != cur.0 { (0usize, 0usize) } else { (1, 1) };
        let k = (1..4usize).find(|&kk| swap_jeu(pred, cote, kk) == cur).unwrap();
        let mur = mur_du_niveau(jeu_tab(pred, ti), k);
        let cout = w[&k];
        cumul += cout;
        if cote == 0 { pl += cout; } else { pc += cout; }
        println!("  pas {} : {}", n + 1, nommer_pas(pred, cur));
        println!("          jeu {}|{} -> {}|{}", py_tup4(pred.0), py_tup4(pred.1), py_tup4(cur.0), py_tup4(cur.1));
        let faces = briser_le_tie(mur);
        println!("          mur traversé : table {} {} (codim 1), faces {}, {}", cote_str(cote), py_tup4(mur), py_tup4(faces.0), py_tup4(faces.1));
        println!("          coût {} payé par {} | cumul {}", cout, cote_str(cote), cumul);
    }
    println!("  {} : Ligne {} | Colonne {}", nom_arr, py_tup4(arrivee.0), py_tup4(arrivee.1));
    println!("  ARRIVÉE : {} pas | facture totale {} (Ligne {}, Colonne {})", chaine.len(), cumul, pl, pc);
    chaine
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let poule: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let cerf: Jeu = ((4, 1, 3, 2), (4, 3, 1, 2));
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let renverse: Jeu = ((4, 3, 2, 1), (4, 3, 2, 1));

let w: BTreeMap<usize, i64> = BTreeMap::from([(1, 1), (2, 2), (3, 3)]);

// === Section 4.1 : parcours_complet -- le bloc du critère d'intégration ===

let chemin_final = parcours_complet(pd, cerf, &w, "Dilemme (PD)", "Chasse au Cerf (CERF)");


PARCOURS COMPLET : Dilemme (PD) -> Chasse au Cerf (CERF)


  barème par niveau : {1: 1, 2: 2, 3: 3} (le sommet coûte le plus cher)


  Dilemme (PD) : Ligne (3, 1, 4, 2) | Colonne (3, 4, 1, 2)


  pas 1 : R3 (côté ligne, niveaux 3 <-> 4)


          jeu (3, 1, 4, 2)|(3, 4, 1, 2) -> (4, 1, 3, 2)|(3, 4, 1, 2)


          mur traversé : table ligne (3, 1, 3, 2) (codim 1), faces (3, 1, 4, 2), (4, 1, 3, 2)


... (3 outputs left out)

          mur traversé : table colonne (3, 3, 1, 2) (codim 1), faces (3, 4, 1, 2), (4, 3, 1, 2)


          coût 3 payé par colonne | cumul 6


  Chasse au Cerf (CERF) : Ligne (4, 1, 3, 2) | Colonne (4, 3, 1, 2)


  ARRIVÉE : 2 pas | facture totale 6 (Ligne 3, Colonne 3)


### Lecture : le récit complet, lisible d'un bloc

Le bloc répond terme à terme au critère. Le jeu de départ porte un nom (Dilemme) et l'arrivée aussi (Chasse au Cerf). Le chemin est **construit** par BFS -- le lecteur ne l'a pas écrit. Chaque pas affiche **son mur** : la table à égalité exactement entre les deux jeux, avec ses deux faces, qui sont les extrémités du pas. Et chaque pas affiche **son coût** : deux révisions de sommet à 3 échelons chacune, payées séparément -- Ligne paie 3, Colonne paie 3, facture totale 6.

Le récit économique se lit maintenant mot à mot : *sortir du Dilemme vers le Cerf ne coûte aucun bas niveau -- il ne coûte que des sommets*. Personne n'a besoin de réviser ses pires cases ; chacun doit seulement admettre que sa meilleure case change de nature. C'est précisément le contenu de GT-3e sur ce couple (l'indifference exacte de la fuite solitaire), désormais visible comme géométrie tarifée : les deux murs franchis sont tous deux des murs de sommet.

In [20]:
use std::cmp::Reverse;
use std::collections::{BTreeMap, BTreeSet, BinaryHeap, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn swap_val(t: Tab, k: usize) -> Tab {
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    let mut l = [t.0, t.1, t.2, t.3];
    l.swap(pk, pk1);
    (l[0], l[1], l[2], l[3])
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    swap_val(t, k)
}

fn swap_jeu(jeu: Jeu, cote: usize, k: usize) -> Jeu {
    // cote 0 = "ligne", cote 1 = "colonne"
    if cote == 0 {
        (swap_valeurs_adjacentes(jeu.0, k), jeu.1)
    } else {
        (jeu.0, swap_valeurs_adjacentes(jeu.1, k))
    }
}

fn jeu_tab(j: Jeu, ti: usize) -> Tab {
    if ti == 0 { j.0 } else { j.1 }
}

fn tab_get(t: Tab, i: usize) -> usize {
    [t.0, t.1, t.2, t.3][i]
}

fn cote_str(cote: usize) -> &'static str {
    if cote == 0 { "ligne" } else { "colonne" }
}

fn nommer_pas(pred: Jeu, cur: Jeu) -> String {
    for cote in 0..2usize {
        if jeu_tab(pred, cote) != jeu_tab(cur, cote) {
            for k in 1..4usize {
                if swap_jeu(pred, cote, k) == cur {
                    let nom_cote = if cote == 0 { "R" } else { "C" };
                    return format!("{}{} (côté {}, niveaux {} <-> {})", nom_cote, k, cote_str(cote), k, k + 1);
                }
            }
        }
    }
    "?".to_string()
}

fn construire_chemin(depart: Jeu, arrivee: Jeu) -> Option<Vec<(Jeu, Jeu)>> {
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    parent.insert(depart, None);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        if u == arrivee { break; }
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if !parent.contains_key(&v) {
                    parent.insert(v, Some(u));
                    q.push_back(v);
                }
            }
        }
    }
    if !parent.contains_key(&arrivee) { return None; }
    let mut chaine = Vec::new();
    let mut cur = arrivee;
    while let Some(Some(p)) = parent.get(&cur) {
        chaine.push((*p, cur));
        cur = *p;
    }
    chaine.reverse();
    Some(chaine)
}

fn bfs_from(t0: Tab) -> BTreeMap<Tab, usize> {
    let mut d = BTreeMap::new();
    d.insert(t0, 0);
    let mut q = VecDeque::new();
    q.push_back(t0);
    while let Some(u) = q.pop_front() {
        let du = d[&u];
        for k in 1..4usize {
            let v = swap_val(u, k);
            if !d.contains_key(&v) {
                d.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    d
}

fn payoff(row: Tab, col: Tab, cote: usize) -> usize {
    let (st, (r, c)) = br_dyn(row, col);
    if st == "cycle" { return 0; }
    if cote == 0 { at(row, r, c) } else { at(col, r, c) }
}

fn br_dyn(row: Tab, col: Tab) -> (&'static str, (usize, usize)) {
    let (mut r, mut c) = (0usize, 0usize);
    for _ in 0..16 {
        let mut moved = false;
        let nr = if at(row, r, c) > at(row, 1 - r, c) { r } else { 1 - r };
        if nr != r { r = nr; moved = true; }
        let nc = if at(col, r, c) > at(col, r, 1 - c) { c } else { 1 - c };
        if nc != c { c = nc; moved = true; }
        if !moved { return ("ok", (r, c)); }
    }
    ("cycle", (r, c))
}

fn at(t: Tab, r: usize, c: usize) -> usize {
    tab_get(t, 2 * r + c)
}

fn mur_du_niveau(t: Tab, k: usize) -> Tab {
    let mut l = [0usize; 4];
    for i in 0..4 {
        l[i] = if tab_get(t, i) > k { tab_get(t, i) - 1 } else { tab_get(t, i) };
    }
    let pk = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k).unwrap();
    let pk1 = [t.0, t.1, t.2, t.3].iter().position(|&x| x == k + 1).unwrap();
    l[pk] = k;
    l[pk1] = k;
    (l[0], l[1], l[2], l[3])
}

fn briser_le_tie(t: Tab) -> (Tab, Tab) {
    // la valeur ex aequo (mur simple : unique)
    let mut v = usize::MAX;
    for i in 0..4 {
        let x = tab_get(t, i);
        if (0..4).any(|j| j != i && tab_get(t, j) == x) { v = v.min(x); }
    }
    let ij: Vec<usize> = (0..4).filter(|&i| tab_get(t, i) == v).collect();
    let (i, j) = (ij[0], ij[1]);
    let mut inc = [0usize; 4];
    for q in 0..4 {
        inc[q] = if tab_get(t, q) > v { tab_get(t, q) + 1 } else { tab_get(t, q) };
    }
    let mut a = inc;
    a[i] = v; a[j] = v + 1;
    let mut b = inc;
    b[i] = v + 1; b[j] = v;
    ((a[0], a[1], a[2], a[3]), (b[0], b[1], b[2], b[3]))
}

fn nb_uniques(t: Tab) -> usize {
    BTreeSet::from([t.0, t.1, t.2, t.3]).len()
}

fn parcours_complet(depart: Jeu, arrivee: Jeu, w: &BTreeMap<usize, i64>, nom_dep: &str, nom_arr: &str) -> Vec<(Jeu, Jeu)> {
    let chaine = construire_chemin(depart, arrivee).unwrap();
    println!("PARCOURS COMPLET : {} -> {}", nom_dep, nom_arr);
    println!("  barème par niveau : {:?} (le sommet coûte le plus cher)", w);
    println!("  {} : Ligne {} | Colonne {}", nom_dep, py_tup4(depart.0), py_tup4(depart.1));
    let mut cumul = 0i64;
    let (mut pl, mut pc) = (0i64, 0i64);
    for (n, &(pred, cur)) in chaine.iter().enumerate() {
        let (cote, ti) = if pred.0 != cur.0 { (0usize, 0usize) } else { (1, 1) };
        let k = (1..4usize).find(|&kk| swap_jeu(pred, cote, kk) == cur).unwrap();
        let mur = mur_du_niveau(jeu_tab(pred, ti), k);
        let cout = w[&k];
        cumul += cout;
        if cote == 0 { pl += cout; } else { pc += cout; }
        println!("  pas {} : {}", n + 1, nommer_pas(pred, cur));
        println!("          jeu {}|{} -> {}|{}", py_tup4(pred.0), py_tup4(pred.1), py_tup4(cur.0), py_tup4(cur.1));
        let faces = briser_le_tie(mur);
        println!("          mur traversé : table {} {} (codim 1), faces {}, {}", cote_str(cote), py_tup4(mur), py_tup4(faces.0), py_tup4(faces.1));
        println!("          coût {} payé par {} | cumul {}", cout, cote_str(cote), cumul);
    }
    println!("  {} : Ligne {} | Colonne {}", nom_arr, py_tup4(arrivee.0), py_tup4(arrivee.1));
    println!("  ARRIVÉE : {} pas | facture totale {} (Ligne {}, Colonne {})", chaine.len(), cumul, pl, pc);
    chaine
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

fn cout_chemin(chaine: &[(Jeu, Jeu)], w: &BTreeMap<usize, i64>) -> i64 {
    let mut total = 0i64;
    for &(pred, cur) in chaine {
        let (cote, _) = if pred.0 != cur.0 { (0usize, 0usize) } else { (1, 1) };
        let k = (1..4usize).find(|&kk| swap_jeu(pred, cote, kk) == cur).unwrap();
        total += w[&k];
    }
    total
}

fn chemin_min_cout(depart: Jeu, arrivee: Jeu, w: &BTreeMap<usize, i64>) -> Vec<(Jeu, Jeu)> {
    let inf = 1_000_000_000i64;
    let mut dist: BTreeMap<Jeu, i64> = BTreeMap::new();
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    let mut pq: BinaryHeap<Reverse<(i64, Jeu)>> = BinaryHeap::new();
    dist.insert(depart, 0);
    parent.insert(depart, None);
    pq.push(Reverse((0, depart)));
    while let Some(Reverse((d, u))) = pq.pop() {
        if u == arrivee { break; }
        if d > *dist.get(&u).unwrap_or(&inf) { continue; }
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if d + w[&k] < *dist.get(&v).unwrap_or(&inf) {
                    dist.insert(v, d + w[&k]);
                    parent.insert(v, Some(u));
                    pq.push(Reverse((d + w[&k], v)));
                }
            }
        }
    }
    let mut chaine = Vec::new();
    let mut cur = arrivee;
    while let Some(Some(p)) = parent.get(&cur) {
        chaine.push((*p, cur));
        cur = *p;
    }
    chaine.reverse();
    chaine
}

fn bfs_complet(depart: Jeu) -> BTreeMap<Jeu, usize> {
    let mut dist = BTreeMap::new();
    dist.insert(depart, 0);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for cote in 0..2usize {
            for k in 1..4usize {
                let v = swap_jeu(u, cote, k);
                if !dist.contains_key(&v) {
                    dist.insert(v, du + 1);
                    q.push_back(v);
                }
            }
        }
    }
    dist
}

fn pas_elementaire_valide(pred: Jeu, cur: Jeu) -> bool {
    (0..2usize).any(|cote| (1..4usize).any(|k| swap_jeu(pred, cote, k) == cur))
}

fn verifier_parcours(depart: Jeu, arrivee: Jeu, chaine: &[(Jeu, Jeu)], w: &BTreeMap<usize, i64>, nom_dep: &str, nom_arr: &str) -> String {
    if chaine.is_empty() || chaine[0].0 != depart || chaine[chaine.len() - 1].1 != arrivee {
        return "INVALIDE : les extrémités ne sont pas celles annoncées".to_string();
    }
    let mut cumul = 0i64;
    for &(pred, cur) in chaine {
        let (cote, ti) = if pred.0 != cur.0 { (0usize, 0usize) } else { (1, 1) };
        let k = match (1..4usize).find(|&kk| swap_jeu(pred, cote, kk) == cur) {
            Some(kk) => kk,
            None => return format!("INVALIDE : pas non élémentaire vers {}", py_jeu(cur)),
        };
        let f = briser_le_tie(mur_du_niveau(jeu_tab(pred, ti), k));
        let mut faces = vec![f.0, f.1];
        faces.sort();
        let mut att = vec![jeu_tab(pred, ti), jeu_tab(cur, ti)];
        att.sort();
        if faces != att {
            return format!("INVALIDE : le mur exhibé ne sépare pas {} et {}", py_jeu(pred), py_jeu(cur));
        }
        cumul += w[&k];
    }
    let d_long = bfs_complet(depart)[&arrivee];
    if chaine.len() != d_long {
        return format!("INVALIDE : {} pas, distance réelle {}", chaine.len(), d_long);
    }
    let opt = chemin_min_cout(depart, arrivee, w);
    let d_cout = cout_chemin(&opt, w);
    if cumul != d_cout {
        return format!("VALIDE mais NON MINIMAL EN COÛT : {} payés, optimum {}", cumul, d_cout);
    }
    format!("TEMOIN CONSTRUIT ET VÉRIFIÉ INDÉPENDAMMENT : {} -> {} | {} pas = distance exhaustive | facture {} = optimum Dijkstra | chaque mur confirmé par ses faces",
        nom_dep, nom_arr, chaine.len(), cumul)
}

fn py_jeu(j: Jeu) -> String {
    format!("({}, {})", py_tup4(j.0), py_tup4(j.1))
}

let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let poule: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let cerf: Jeu = ((4, 1, 3, 2), (4, 3, 1, 2));
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let renverse: Jeu = ((4, 3, 2, 1), (4, 3, 2, 1));

let w: BTreeMap<usize, i64> = BTreeMap::from([(1, 1), (2, 2), (3, 3)]);

let chemin_final = construire_chemin(pd, cerf).unwrap();

// === Section 4.2 : verifier_parcours -- la re-vérification indépendante, étendue ===

println!("{}", verifier_parcours(pd, cerf, &chemin_final, &w, "Dilemme (PD)", "Chasse au Cerf (CERF)"));
let voisin_cerf47 = swap_jeu(cerf, 0, 1);
let mut chaine47 = chemin_final.clone();
chaine47.push((cerf, voisin_cerf47));
chaine47.push((voisin_cerf47, cerf));
println!();
println!("Contre-épreuve -- le même parcours rallongé d'un aller-retour gratuit pour les yeux :");
println!("   {}", verifier_parcours(pd, cerf, &chaine47, &w, "PD", "CERF"));


TEMOIN CONSTRUIT ET VÉRIFIÉ INDÉPENDAMMENT : Dilemme (PD) -> Chasse au Cerf (CERF) | 2 pas = distance exhaustive | facture 6 = optimum Dijkstra | chaque mur confirmé par ses faces


Contre-épreuve -- le même parcours rallongé d'un aller-retour gratuit pour les yeux :


   INVALIDE : 4 pas, distance réelle 2


### Lecture : le témoin construit et vérifié indépendamment

Le vérificateur étendu ne se contente pas des extrémités et de l'élémentarité : il re-dérive **chaque mur** (brisure de tie recalculée, faces comparées aux extrémités affichées), **chaque coût** (barème réappliqué pas à pas), la **minimalité en longueur** (BFS exhaustif) et la **minimalité en coût** (Dijkstra indépendant) -- et rend son verdict en une ligne. La contre-épreuve finale montre qu'un chemin rallongé d'un aller-retour, invisible aux yeux s'il était bien imprimé, est attrapé par la distance réelle.

La terminologie est celle du steer #12205 : un chemin produit par un constructeur et accepté par un vérificateur séparé est un **témoin construit et vérifié indépendamment** -- pas une preuve au sens formel, mais un objet dont chaque affirmation affichée a été re-dérivée par un code qui ne partage rien avec celui qui l'a produite. C'est le standard de la série depuis GT-20, étendu ici aux murs et aux coûts.

## Exercice 1 -- le sweep vu par Colonne

La marche 2 a fait migrer Ligne. Refaites le meme sweep du cout **cote Colonne** (ses swaps C1-C3, son gain au sorti de la dynamique). Le compte de migrants a c=1 est-il le meme que cote Ligne (16 %) ? Si oui, dites pourquoi la symetrie l'impose ; sinon, exhibez un jeu qui les distingue.

In [21]:
use std::collections::BTreeMap;

// Exercice 1 : sweep du cout cote Colonne
// TODO etudiant : reconstruire best_by_dist_col (BFS sur la table Colonne, payoff cote 1)
// puis le tableau c=0..3 : %migrer, distance optimale moyenne, fuient-un-cycle
let resultat_ex1: Option<BTreeMap<usize, (usize, f64, usize)>> = None;  // TODO etudiant : {c: (pct_migrer, dist_moyenne, fuites)}
println!("Exercice 1 : a completer (sweep cote Colonne)");


Exercice 1 : a completer (sweep cote Colonne)


## Exercice 2 -- deplacer la selection d'equilibre

Les 72 jeux a deux equilibres purs sont les coordinations. En partant de l'un d'eux, montrez un cas ou **un seul swap de Ligne change l'equilibre selectionne** par la dynamique (la case d'arrivee change), et un cas ou aucun swap ne la change. Combien des 72 jeux sont du premier type ? (Le comptage exhaustif est possible : 72 jeux x 24 tables.)

In [22]:
type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

// Exercice 2 : la selection d'equilibre sous meta-actions
// TODO etudiant : pour chaque jeu a 2 NE purs, comparer l'arrivee de br_dyn pour la table
// courante et pour les 23 autres tables ; compter ceux ou l'arrivee peut changer
let resultat_ex2: Option<(usize, Jeu)> = None;  // TODO etudiant : (nb_jeux_selection_deplacable, exemple)
println!("Exercice 2 : a completer (selection deplacable)");


Exercice 2 : a completer (selection deplacable)


## Exercice 3 -- le meta-jeu sequentiel

La marche 3 est simultanee : les 4 echecs de coordination dur proviennent de l'instabilite de l'accord. Rendez le meta-jeu **sequentiel** : Ligne annonce et joue sa meta-action, Colonne voit le jeu deplace puis choisit la sienne. Proposez le protocole (qui observe quoi, ordre des couts), recalculez les equilibres (backward induction sur les 4x4 sous-jeux), et dites combien des 4 echecs durs se referment. Verdict attendu en une phrase : *le sequentiel suffit-il a stabiliser l'accord ?*

In [23]:
use std::collections::BTreeMap;

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

// Exercice 3 : equilibres parfaits du meta-jeu sequentiel
// TODO etudiant : pour chacun des 4 jeux en echec dur, construire l'arbre
// (Ligne : 4 actions ; Colonne observe ; 4 actions) et resoudre par induction
let resultat_ex3: Option<BTreeMap<Jeu, ((usize, usize), bool)>> = None;  // TODO etudiant : {jeu: equilibre_sequentiel, referme: bool}
println!("Exercice 3 : a completer (meta-jeu sequentiel)");


Exercice 3 : a completer (meta-jeu sequentiel)


## Exercice 4 — Le mur du pas, dans l'autre sens

La section 2 allait du pas vers le mur ; cet exercice remonte le chemin inverse. On donne deux chambres adjacentes quelconques `(pred, cur)` et on demande de retrouver le mur traversé : identifier le côté modifié et le niveau `k` (le pas), construire le mur par fusion de niveaux, puis vérifier que `briser_le_tie(mur)` redonne exactement `{pred, cur}` — la propriété de la section 2.

Test attendu sur `pred_ex -> cur_ex` : côté colonne, niveau 1, mur `(2, 3, 1, 1)`.

In [24]:
type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn py_jeu(j: Jeu) -> String {
    format!("({}, {})", py_tup4(j.0), py_tup4(j.1))
}

fn py_tup4(t: Tab) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

// Exercice 1 -- le mur du pas, dans l'autre sens
// On donne deux chambres adjacentes quelconques (pred, cur). Retrouver le mur traversé :
// 1. identifier le côté modifié et le niveau k (le pas) ;
// 2. construire le mur par fusion de niveaux ;
// 3. vérifier que briser_le_tie(mur) redonne exactement {pred, cur} -- la propriété de la section 2.
// Test attendu sur pred_ex -> cur_ex : côté colonne, niveau 1, mur (2, 3, 1, 1).
fn mur_du_pas(_pred: Jeu, _cur: Jeu) -> Option<(usize, usize, Tab)> {
    // Retourne (cote, k, mur) du pas pred -> cur, ou None si non adjacent.
    // Etape 1 : identifier le cote modifie et le niveau k (le pas)
    // Etape 2 : construire le mur par fusion de niveaux
    // Etape 3 : briser_le_tie(mur) doit redonner exactement {pred, cur}
    // TODO étudiant
    None  // a remplacer
}
let pred_ex: Jeu = ((2, 1, 4, 3), (3, 4, 1, 2));
let cur_ex: Jeu = ((2, 1, 4, 3), (3, 4, 2, 1));
println!("Exercice 1 à compléter : mur_du_pas {} -> {}", py_jeu(pred_ex), py_jeu(cur_ex));


Exercice 1 à compléter : mur_du_pas ((2, 1, 4, 3), (3, 4, 1, 2)) -> ((2, 1, 4, 3), (3, 4, 2, 1))


## Exercice 5 — Chercher le contre-exemple au fait mesuré

La section 3.2 a **mesuré** (10 paires nommées + 300 paires quelconques) que le chemin le plus court est aussi le moins cher sous `W = {1: 1, 2: 2, 3: 3}`. Le fait est-il robuste à un autre barème ? Choisir un autre tarif (p.ex. `W2 = {1: 1, 2: 1, 3: 3}` ou `{1: 3, 2: 1, 3: 1}`, le sommet bon marché), refaire le balayage — pour chaque paire nommée et un échantillon de 300 paires, comparer `cout_chemin(construire_chemin)` et `cout_chemin(chemin_min_cout)` — et rapporter : un barème où un détour gagne existe-t-il ?

Indice : commencer par le barème sommet bon marché — que devient l'écart moyen ?

In [25]:
use std::collections::BTreeMap;

// Exercice 2 -- chercher le contre-exemple au fait mesuré
// La section 3.2 a MESURÉ (10 paires nommées + 300 paires quelconques) que le chemin le plus court
// est aussi le moins cher sous W = {1: 1, 2: 2, 3: 3}. Est-ce robuste à un autre barème ?
// 1. choisir un autre tarif, p.ex. W2 = {1: 1, 2: 1, 3: 3} ou {1: 3, 2: 1, 3: 1} (sommet bon marché !) ;
// 2. re-faire le balayage : pour chaque paire nommée et un échantillon de 300 paires,
//    comparer cout_chemin(construire_chemin) et cout_chemin(chemin_min_cout) ;
// 3. rapporter : un barème où un détour gagne existe-t-il ?
// Indice : commencer par le barème sommet bon marché -- que devient l'écart moyen ?
let w2: BTreeMap<usize, i64> = BTreeMap::from([(1, 1), (2, 2), (3, 3)]);  // TODO etudiant : essayer d'autres barèmes
println!("Exercice 2 à compléter : balayage sous barème {:?}", w2);


Exercice 2 à compléter : balayage sous barème {1: 1, 2: 2, 3: 3}


## Exercice 6 — Capstone : POULE -> RENVERSE, le parcours complet

Faire courir `parcours_complet` de la Poule au Renversement sous `W = {1: 1, 2: 2, 3: 3}`, puis passer le résultat à `verifier_parcours`. Questions de lecture : combien de pas, et répartis comment entre les deux joueurs ? Quels niveaux sont révisés — le trajet touche-t-il le sommet de qui que ce soit ? La facture par joueur : qui paie le plus, et pourquoi la géométrie l'impose-t-elle (cf 3.1) ?

In [26]:
// Exercice 3 -- capstone : POULE -> RENVERSE, le parcours complet
// Faire courir parcours_complet de la Poule au Renversement sous W = {1: 1, 2: 2, 3: 3},
// puis passer le résultat à verifier_parcours. Questions de lecture :
// 1. combien de pas, et répartis comment entre les deux joueurs ?
// 2. quels niveaux sont révisés -- le trajet touche-t-il le sommet de qui que ce soit ?
// 3. la facture par joueur : qui paie le plus, et pourquoi la géométrie l'impose-t-elle (cf 3.1) ?
println!("Exercice 3 à compléter : parcours POULE -> RENVERSE + vérification");


Exercice 3 à compléter : parcours POULE -> RENVERSE + vérification


## 9. Preuve formelle — le module `Abstraction/Basic.lean`, consommé aux lignes mesurées

Tout ce carnet a **mesuré** le prix des méta-actions : seuils de migration (marche 2), tarifs par niveau (section 7), et ce fait fragile du 3.2 — le plus court chemin est aussi le moins cher, *sur les paires testées*. Le lac `game_theory_lean` du dépôt porte un module qui pousse cette question un étage plus haut dans la formalisation : `Abstraction/Basic.lean` (grain #12204) certifie par le noyau Lean ce qui arrive quand la méta-action n'est plus *déplacer le jeu* mais **raffiner l'abstraction sous laquelle on joue** — et sa réponse est une réfutation. Le module est **volontairement sans Mathlib** (pattern `Swaps`, cf [GT-3b, section 10-bis](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb)) : tout s'y clôt par `decide`/`rfl` sur des littéraux entiers.

Comme en 10-bis de GT-3b, la consommation se fait **sans lancer `lake` en sous-processus** : la cellule suivante lit le fichier source depuis ce dépôt, extrait chaque déclaration **à sa ligne réelle**, et vérifie la propreté axiomatique sur les lignes de code seules (docstrings et commentaires retirés). Les citations de la section s'appuient sur ces numéros, pas sur une mémoire.

In [27]:
fn trouver_abstraction_lean() -> Option<(std::path::PathBuf, &'static str)> {
    let mut base = std::env::current_dir().ok()?;
    loop {
        for rel in ["game_theory_lean/Abstraction/Basic.lean",
                    "MyIA.AI.Notebooks/GameTheory/game_theory_lean/Abstraction/Basic.lean"] {
            let cible = base.join(rel);
            if cible.exists() { return Some((cible, rel)); }
        }
        if !base.pop() { return None; }
    }
}

const DECL_KINDS: [&str; 8] = ["theorem", "lemma", "def", "noncomputable def", "abbrev", "structure", "inductive", "instance"];

fn scan_decl(ligne: &str) -> Option<(&'static str, String)> {
    for kind in DECL_KINDS {
        let rest = match ligne.strip_prefix(kind) { Some(r) => r, None => continue };
        let n_ws = rest.chars().take_while(|c| *c == ' ' || *c == '\t').count();
        if n_ws == 0 { continue; }
        let after = &rest[n_ws..];
        let mut chars = after.chars();
        let first = chars.next()?;
        if !(first.is_ascii_alphabetic() || first == '_') { continue; }
        let name: String = std::iter::once(first)
            .chain(chars.take_while(|c| c.is_ascii_alphanumeric() || *c == '_' || *c == '\''))
            .collect();
        if name.is_empty() { continue; }
        return Some((kind, name));
    }
    None
}

fn find_close(s: &[char], from: usize) -> Option<usize> {
    let mut j = from;
    while j + 1 < s.len() {
        if s[j] == '-' && s[j + 1] == '/' { return Some(j); }
        j += 1;
    }
    None
}

fn strip_lean_noise(text: &str) -> String {
    // passe 1 : /- ... -/ (le plus court, non chevauchant, reprise apres echec)
    let s: Vec<char> = text.chars().collect();
    let mut out: Vec<char> = Vec::new();
    let mut i = 0;
    while i < s.len() {
        if s[i] == '/' && i + 1 < s.len() && s[i + 1] == '-' {
            if let Some(cl) = find_close(&s, i + 2) {
                out.push(' ');
                i = cl + 2;
                continue;
            }
        }
        out.push(s[i]);
        i += 1;
    }
    // passe 2 : -- jusqu'a fin de ligne (exclue)
    let s2 = out;
    let mut out2: Vec<char> = Vec::new();
    let mut i = 0;
    while i < s2.len() {
        if s2[i] == '-' && i + 1 < s2.len() && s2[i + 1] == '-' {
            while i < s2.len() && s2[i] != '\n' { i += 1; }
            out2.push(' ');
            continue;
        }
        out2.push(s2[i]);
        i += 1;
    }
    out2.into_iter().collect()
}

fn is_word_char(c: char) -> bool { c.is_alphanumeric() || c == '_' }

fn count_word(text: &str, mot: &str) -> usize {
    let chars: Vec<char> = text.chars().collect();
    let mot: Vec<char> = mot.chars().collect();
    let mut count = 0;
    let mut i = 0;
    while i + mot.len() <= chars.len() {
        if chars[i..i + mot.len()] == mot[..] {
            let before_ok = i == 0 || !is_word_char(chars[i - 1]);
            let after_ok = i + mot.len() >= chars.len() || !is_word_char(chars[i + mot.len()]);
            if before_ok && after_ok { count += 1; i += mot.len(); continue; }
        }
        i += 1;
    }
    count
}

fn rel_affiche(chemin: &std::path::Path) -> String {
    let comps: Vec<String> = chemin.components()
        .map(|c| c.as_os_str().to_string_lossy().to_string())
        .collect();
    let n = comps.len();
    comps[n - 3..].join("\\")
}

// === Preuve formelle : localiser Abstraction/Basic.lean, extraire, verifier la proprete ===
// Patron GT-3b (5-bis) et GT-6b (cellule 2) : lecture du source, jamais de lake en sous-processus.

match trouver_abstraction_lean() {
    None => println!("module Abstraction/Basic.lean introuvable -- cloner le depot complet pour lire les citations"),
    Some((chemin_module, _rel)) => {
        let src = std::fs::read_to_string(&chemin_module).unwrap();
        let lignes: Vec<&str> = src.lines().collect();
        let code = strip_lean_noise(&src);
        let mut decls: Vec<(usize, &'static str, String)> = Vec::new();
        for (i, l) in lignes.iter().enumerate() {
            if let Some((k, n)) = scan_decl(l) {
                decls.push((i + 1, k, n));
            }
        }
        let n_thm = decls.iter().filter(|(_, k, _)| *k == "theorem" || *k == "lemma").count();
        println!("{} : {} lignes, {} declarations dont {} theoremes", rel_affiche(&chemin_module), lignes.len(), decls.len(), n_thm);
        println!();
        let cles = ["encadrement", "dette_nonneg", "selle_dette_nulle",
                    "selle_blocQ", "selle_blocP", "raffinement_aggrave_la_dette", "courbe_dette"];
        println!("declarations portees par le sujet de ce carnet :");
        for (i, k, n) in &decls {
            if cles.contains(&n.as_str()) {
                println!("  L{:>4} [{:<8}] {}", i, k, n);
            }
        }
        println!();
        println!("proprete axiomatique, comptee sur les lignes de code (prose retiree) :");
        for mot in ["sorry", "sorryAx", "native_decide", "axiom"] {
            println!("  occurrences de '{}' : {}", mot, count_word(&code, mot));
        }
    }
};


game_theory_lean\Abstraction\Basic.lean : 309 lignes, 45 declarations dont 21 theoremes


declarations portees par le sujet de ce carnet :


  L 128 [theorem ] encadrement


  L 139 [theorem ] dette_nonneg


  L 145 [theorem ] selle_dette_nulle


  L 226 [theorem ] selle_blocQ


... (4 outputs left out)

  occurrences de 'sorry' : 0


  occurrences de 'sorryAx' : 0


  occurrences de 'native_decide' : 0


  occurrences de 'axiom' : 0


### Lecture du résultat : l'inventaire est compté, la dette axiomatique nulle

Le module est réel et mesuré : l'inventaire imprimé ci-dessus couvre tout le fichier — définitions du modèle, lois générales, contre-exemple — et les déclarations que ce carnet mobilise y sont citées à leurs lignes réelles : les lois générales `encadrement`, `dette_nonneg`, `selle_dette_nulle`, les selles de blocs `selle_blocQ` et `selle_blocP`, le contre-exemple-phare `raffinement_aggrave_la_dette` et son résumé numérique `courbe_dette`. La propreté se compte sur les **lignes de code seules** — l'en-tête du module parle *de* `sorry` en prose, c'est précisément ce que le retrait des docstrings neutralise : zéro `sorry`, zéro `sorryAx`, zéro `native_decide`, zéro `axiom` déclaré. L'en-tête certifie de plus que le seul axiome traversant les lemmes d'ordre généraux de `Int` est `propext`, dont aucun théorème concret ne dépend — tout se clôt par réduction du noyau (`decide`/`rfl`), exactement comme `Swaps` en 5-bis de GT-3b.

In [28]:
type Duel = (i64, i64, i64, i64);

fn gain(m: Duel, i: bool, j: bool) -> i64 {
    match (i, j) { (false, false) => m.0, (false, true) => m.1, (true, false) => m.2, (true, true) => m.3 }
}

fn v_haut(m: Duel, j: bool) -> i64 {
    gain(m, false, j).max(gain(m, true, j))
}

fn v_bas(m: Duel, i: bool) -> i64 {
    gain(m, i, false).min(gain(m, i, true))
}

fn dette(m: Duel, s: (bool, bool)) -> i64 {
    v_haut(m, s.1) - v_bas(m, s.0)
}

fn est_selle(m: Duel, s: (bool, bool)) -> bool {
    let g = gain(m, s.0, s.1);
    g == v_haut(m, s.1) && g == v_bas(m, s.0)
}

fn somme_bloc(duels: &[Duel]) -> Duel {
    let mut out = (0i64, 0, 0, 0);
    for &m in duels {
        out = (out.0 + m.0, out.1 + m.1, out.2 + m.2, out.3 + m.3);
    }
    out
}

fn dette_bloc(duels: &[Duel], s: (bool, bool)) -> i64 {
    duels.iter().map(|&m| dette(m, s)).sum()
}

fn py_tup4i(t: Duel) -> String {
    format!("({}, {}, {}, {})", t.0, t.1, t.2, t.3)
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

// Les trois etats du contre-exemple (L163-L169)
let d0: Duel = (1, 0, 0, -1);
let d1: Duel = (-2, 0, 0, 1);
let d2: Duel = (2, 0, -1, 0);
let bloc_q = somme_bloc(&[d0, d1, d2]);
let bloc_p = somme_bloc(&[d1, d2]);
println!("blocQ = {} -- blocQ_valeur L202 | blocP = {} -- blocP_valeur L205", py_tup4i(bloc_q), py_tup4i(bloc_p));
let s_q = (false, true);
let s_p = (false, false);
let det_q = dette_bloc(&[d0, d1, d2], s_q);
let det_p = dette(d0, (false, true)) + dette_bloc(&[d1, d2], s_p);
let det_d = dette(d0, (false, true)) + dette(d1, (true, false)) + dette(d2, (false, true));
println!("dette retransportee, partition grossiere Q : {}", det_q);
println!("dette retransportee, partition raffinee  P : {}", det_p);
println!("dette retransportee, partition discrete  D : {}", det_d);
println!("selles d'existence verifiees par le miroir : blocQ {} (L226), blocP {} (L236) -- l'unicite est certifiee en L229/L239",
    py_bool(est_selle(bloc_q, s_q)), py_bool(est_selle(bloc_p, s_p)));
println!();
println!("courbe de courbe_dette (L306), reproduite par le miroir : 0 < {} < {}, discret = {}", det_q, det_p, det_d);
assert!(0 < det_q && det_q < det_p && det_d == 0, "le miroir doit reproduire la courbe certifiee par le noyau");
println!("miroir et noyau disent la meme courbe : le contre-exemple est reproduce, chiffre par chiffre");


blocQ = (1, 0, -1, 0) -- blocQ_valeur L202 | blocP = (0, 0, -1, 1) -- blocP_valeur L205


dette retransportee, partition grossiere Q : 3


dette retransportee, partition raffinee  P : 4


dette retransportee, partition discrete  D : 0


selles d'existence verifiees par le miroir : blocQ True (L226), blocP True (L236) -- l'unicite est certifiee en L229/L239


courbe de courbe_dette (L306), reproduite par le miroir : 0 < 3 < 4, discret = 0


miroir et noyau disent la meme courbe : le contre-exemple est reproduce, chiffre par chiffre


### Lecture du résultat : le miroir exécuté dit 0 < 3 < 4, discret 0

Le miroir Rust reproduit chiffre par chiffre la courbe certifiée : le duel abstrait du bloc grossier vaut `(1, 0, -1, 0)` et celui du bloc raffiné `(0, 0, -1, 1)` — exactement les valeurs closes par `rfl` (`blocQ_valeur`, `blocP_valeur`) — et la dette retransportée passe de **3** (partition grossière : la selle `(0,1)` du bloc unique coûte 0 à l'état 0, 3 à l'état 1, 0 à l'état 2) à **4** (partition raffinée : selle propre de {0} gratuite, selle `(0,0)` du bloc {1,2} coûtant 2 + 2), avant de retomber à **0** dans le discret. Le `assert` final échouerait si le miroir divergeait de la courbe du noyau : c'est une vérification, pas une décoration.

### Ce que le module certifie — et ce qu'il réfute

Le pont avec ce carnet est **méthodologique, et il faut le dire** : le module ne calcule pas dans le substrat ordinal (tables de rangs 1-4) des marches 1-3, mais dans des duels 2×2 à somme nulle sur les entiers — la dette qu'il certifie n'est pas un chiffre mesuré ici. Ce qu'il apporte tient en deux temps.

**D'abord une formalisation de la marche 1.** Les trois lois générales disent, dans le modèle du module, pourquoi la dynamique de meilleure réponse ne peut se reposer qu'à l'équilibre pur : le gain de tout profil est encadré entre `vBas` et `vHaut` (`encadrement`), sa dette — l'écart exploitable qui reste — est non négative (`dette_nonneg`), et une selle pure, exactement un point de repos de la dynamique, a dette nulle (`selle_dette_nulle`). C'est la version certifiée du croisement mesuré en marche 1 : convergence si et seulement si équilibre pur.

**Ensuite une réfutation — et elle contredit une intuition que ce carnet aurait pu être tenté de généraliser.** La méta-action du module est *raffiner l'abstraction du jeu* : découper les blocs d'états sous lesquels on joue une selle commune. Le théorème-phare (`raffinement_aggrave_la_dette`) certifie que sur ses trois duels, raffiner {{0,1,2}} en {{0},{1,2}} **aggrave strictement** la dette retransportée, de 3 à 4, quel que soit le choix des selles — elles sont uniques (`selle_blocQ_unique`, `selle_blocP_unique`) — avant que le discret ne retombe à 0 : plus fin n'est pas toujours moins cher. La non-croissance de la dette observée par le notebook GT-19 sur *ses* matrices n'est donc pas un théorème, c'est une propriété de ses données. La leçon retombe directement sur la section 3.2 : « le plus court chemin est aussi le moins cher » y est un **fait mesuré** sur des paires testées, pas une loi, et le module montre sur un cousin de cette intuition ce qu'un contre-exemple certifié fait à une généralisation prématurée. La frontière entre théorème (3.1, la décomposition du coût) et mesure (3.2) traverse tout ce carnet ; le module la garde vivante en rappelant qu'une intuition mesurée n'est pas une loi certifiée.

## Conclusion du parcours complet

Le critère d'intégration du chantier est tenu, bout en bout et sur un seul écran : **un jeu nommé** (le Dilemme), **un chemin non écrit à la main** (BFS + parents, vérifié), **des murs visibles** (chaque pas exhibe la table à égalité entre les deux jeux, ses deux faces confirmées), **des coûts lus** (deux révisions de sommet, 3 échelons chacune, payées séparément) -- et l'arrivée **porte un nom** (la Chasse au Cerf).

Deux résultats structurent le parcours au-delà de l'assemblage. Le **théorème des bornes** (3.1) : le coût du trajet minimal se décompose en `c_L·d_L + c_C·d_C` où `d_L`, `d_C` sont des distances de tables individuelles -- la facture est un invariant géométrique, hors de portée de tout marchandage sur le choix du chemin. Et le **fait mesuré** (3.2) : sous tarification par niveau, le plus court chemin est aussi le moins cher, sur toutes les paires testées -- l'intuition du détour économique ne survit pas à la mesure.

**Pour aller plus loin** : l'arbitrage *migrer ou rester* (à quel prix un agent accepte-t-il de payer ces 6 échelons ?) est le sujet de la Partie I (marche 2, sections 2-3 ci-dessus) ; la théorie des murs et des chambres, celui de [GT-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) ; la séparation constructeur / vérificateur, celle de [Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb).

## Conclusion générale : trois marches, puis un parcours

Les trois marches mesurent le passage de la strate 6 a la strate 7 sur un univers fini ou tout se verifie :

| Marche | Ce que fait l'agent | Chiffre cle |
|---|---|---|
| 1 -- jouer | subit les regles, converge ou cycle | 72 jeux injouables sur 576 ; convergence si et seulement si equilibre pur |
| 2 -- deplacer | paie des echelons pour reecrire sa table | 56 % -> 16 % -> 8 % -> 4 % de migrants quand le cout monte ; le Dilemme exactement indifferent a c=1 |
| 3 -- se coordonner sur les regles | joue le meta-jeu 4x4 | 106 jeux ou bouger est necessaire ; le Dilemme s'evade a (3,3) par equilibre symetrique ; 4 echecs de coordination dur |

Le resultat qui donne son sens au chantier : **changer les regles est une action ordinaire** -- elle a un prix, un seuil, des equilibres, et meme ses propres echecs de coordination. Le vocabulaire strategique de la strate 6 s'applique sans modification un etage plus haut ; c'est la definition meme d'un franchissement reussi.

*Suite du chantier (#12207)* : D1 (la grammaire des generateurs manipulables) et D3 (le chemin minimal certifie, cible Lean a terme via #12205) restent a ecrire sur le meme substrat.

## À retenir

| Leçon | Mesure dans ce carnet |
|---|---|
| Jouer est une dynamique, pas un arbitrage ponctuel | `NE purs par jeu : {0: 72, 1: 432, 2: 72} sur 576` et `Dynamique BR depuis TG : converge 504`, `cycle 72` ; le croisement `{(0, 'cycle'): 72, (1, 'ok'): 432, (2, 'ok'): 72}` établit que converger équivaut à posséder un équilibre pur |
| Changer les règles a un prix, et le prix sélectionne | le sweep fait tomber la part de migrants de `56%` sur la première ligne (`c = 0`) à `16%`, `8%` puis `4%` sur les trois lignes suivantes, avec une distance optimale moyenne qui descend de `1.33` à `1.25` puis `1.00` |
| Le gain net des migrants n'est pas monotone en le coût | `+1.93` à `c = 0`, `+2.00` à `c = 1`, puis `+1.50` à `c = 2` et `+1.00` sur la dernière ligne : la population migrante se concentre d'abord sur les gains forts, ensuite le prix prélevé fait baisser la moyenne |
| Le Dilemme est exactement indifférent à `c = 1` | rang courant `2` ; à distance `1` on atteint le rang `3` (net `2`), à distance `2` le rang `4` (net `2`), à distance `3` le net retombe à `1` ; verdict affiché : à `c = 0`, `Ligne MIGRE a distance 2` ; à `c = 1` comme à `c = 2`, `Ligne RESTE` |
| Partir, c'est parfois fuir l'injouable | `Jeux injouables : 72`, `reparables en 1 swap : 48`, `autres : 24` ; à `c = 1`, `Migrants a c=1 : 96 jeux sur 576 dont 72 refugies` |
| Le mur est une égalité de rangs, pas une frontière de cases | à `lambda = 0.50`, `MUR : u_TG = u_BG -- codimension 1` ; `Murs simples distincts par joueur : 36` ; `Propriété mur <-> paire de chambres, vérifiée sur 24 tables x 3 niveaux : True` |
| Le méta-jeu a lui aussi ses équilibres | `Jeux avec au moins un meta-NE pur :` et `572 / 576` ; `aucun meta-NE ne reste sur place : 106` (bouger est alors nécessaire) contre `134` jeux où tous les équilibres restent en place |
| La coordination peut échouer, et l'échec se compte | `Jeux (avec meta-NE) dont AU MOINS UN equilibre est Pareto-optimal : 568` et `Jeux ou AUCUN equilibre n'est Pareto-optimal : 4` |
| Le chemin le plus court est aussi le moins cher | BFS et Dijkstra coïncident sur toutes les paires nommées (`écart 0`), et `Paires nommées où un détour battrait le plus court chemin : 0 / 10` |
| Le coût est géométrique : il ne se négocie pas | `Tarifs par côté : Ligne 1, Colonne 3` ; la ligne `PD -> CERF` du tableau porte d_L 1, d_C 1, longueur 2 et coût 4, quand la ligne `PD -> IDENTITE` porte d_L 3, d_C 4, longueur 7 et coût 15 |
| Le témoin se construit et se vérifie séparément | `VALIDE + MINIMAL : 2 pas = distance exhaustive d(G,H)` pour le chemin du constructeur, `INVALIDE : pas non élémentaire` pour un pas vers un jeu non adjacent |
| Le formel certifie la dette, il ne la calcule pas | `dette retransportee, partition grossiere Q : 3`, `dette retransportee, partition raffinee  P : 4`, `dette retransportee, partition discrete  D : 0`, et la courbe `0 < 3 < 4, discret = 0` reproduite par le miroir Rust |

**Ce que ce carnet ajoute à la série** : il ne se contente pas de constater qu'un agent peut réécrire les règles — il en mesure le **prix**, le **seuil**, l'**équilibre** et l'**échec de coordination**, puis il referme la boucle par un parcours dont chaque pas est exhibé (jeu de départ, jeu d'arrivée, mur traversé, faces du mur, coût payé) et dont le témoin est vérifié par un vérificateur indépendant. Le versant formel (`game_theory_lean\Abstraction\Basic.lean`) est consommé à la ligne près : `selle_blocQ` (L226), `selle_blocP` (L236), `courbe_dette` (L306), avec une impureté axiomatique comptée à `0`.

## Vérifiez votre compréhension

**1. Le sweep fait tomber la part de migrants de `56%` sur sa première ligne à `4%` sur la dernière, et le gain net moyen des migrants *augmente* d'abord : `+1.93` à `c = 0`, `+2.00` à `c = 1`. Un coût plus élevé devrait au contraire réduire le gain de ceux qui migrent. Que se passe-t-il ?**

<details><summary>Réponse</summary>

Le gain net moyen est une moyenne **conditionnelle** : elle porte sur la population qui migre, et cette population change avec le prix. À `c = 0`, `56%` des jeux vues ont un agent qui migre, y compris des cas où l'écart de rang est faible : la moyenne est diluée (`+1.93`, distance optimale moyenne `1.33`). À `c = 1`, seuls `16%` migrent — le prix a éliminé les gains marginaux, et il reste ceux qui ont le plus à perdre au jeu courant : la moyenne monte (`+2.00`) malgré un coût ajouté. Ensuite le gain net redescend (`+1.50` à `c = 2`, puis `+1.00` sur la dernière ligne), non plus par sélection mais parce que le net est un net : le même gain brut, amputé du prix payé. En clair, le prix agit **deux fois** — il filtre les migrants, puis il prélève sur eux.

</details>

**2. Le tableau des tarifs par côté annonce un coût de 4 pour `PD -> CERF`, et le parcours complet annonce une `facture totale 6 (Ligne 3, Colonne 3)` pour le même trajet. Lequel des deux chiffres est le bon ?**

<details><summary>Réponse</summary>

Les deux, parce qu'ils ne tarifent pas la même chose. Le tableau de la section « tarifs par côté » facture des **déplacements de tables** : le coût y vaut `c_L·d_L + c_C·d_C`, avec `c_L` valant 1 et `c_C` valant 3 (`Tarifs par côté : Ligne 1, Colonne 3`), ce qui donne pour `PD -> CERF` d_L 1 et d_C 1, soit 1 + 3 = 4. Le barème du parcours complet facture des **niveaux** : `{1: 1, 2: 2, 3: 3}`, et les deux pas du parcours (un échange de niveaux `3 <-> 4` de chaque côté) traversent donc le niveau `3` à chaque fois, soit 3 + 3 = 6. C'est la même géométrie, deux modèles de prix : l'un compte les cases franchies, l'autre les niveaux traversés. Le carnet ne les mélange jamais, et c'est ce qui rend la remarque « le plus court est aussi le moins cher » (`écart 0` entre BFS et Dijkstra sur toutes les paires nommées) vérifiable plutôt que credite.

</details>

## Pour aller plus loin

- [README de la série GameTheory](README.md) : la place du carnet dans l'arc « les jeux comme objets » — le versant où l'agent achète le droit de réécrire les règles, avec un prix et un seuil.
- Les six exercices du carnet rejouent la mécanique plutôt que de la recopier : le sweep côté Colonne (exercice 1), la sélection d'équilibre déplaçable (exercice 2), le méta-jeu séquentiel (exercice 3), le mur du pas dans l'autre sens (exercice 4), la recherche du contre-exemple au fait mesuré (exercice 5), et le capstone `POULE -> RENVERSE` (exercice 6) qui rejoue le parcours complet sur une autre paire.
- Le module `game_theory_lean\Abstraction\Basic.lean` (`45 declarations dont 21 theoremes`) porte la version formelle du versant dette : `encadrement` (L128), `dette_nonneg` (L139), `selle_dette_nulle` (L145), `selle_blocQ` (L226), `selle_blocP` (L236), `raffinement_aggrave_la_dette` (L294), `courbe_dette` (L306). Le carnet en consomme l'inventaire et en rejoue la courbe par un miroir Rust.
- Les carnets voisins prolongent la même marche : GT-3b pour la théorie des murs et des chambres, GT-20 pour la séparation constructeur / vérificateur des chemins, GT-21 pour les deux espèces de flèches sur le même espace.

***

[← GameTheory-03d-Deux-Especes-de-Fleches-Python](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) | [↑ README GameTheory](README.md) | [jumeau Python](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb)

*GameTheory 3e -- Meta-Actions Tarifees, versant D4 du chantier « Les jeux comme objets » (#12207). Substrat, encodages et swaps herites de GT-3b ; jumeaux conceptuels GT-21 (les deux especes de fleches) et GT-20 (l'engagement comme protocole de coordination sur les regles).*